# Graph Databases with Neo4j & Cypher
## A Hands-On Course — From Zero to Intermediate

**Dataset:** a real e-commerce business — customers, orders, products, categories, suppliers, warehouses, reviews, shipments and payments.

---

### How to use this notebook

This is a *workshop*, not a reference manual. Every section follows the same rhythm:

| Step | What happens |
|---|---|
| **Concept** | A short explanation of *why* the idea exists, before any syntax |
| **Diagram** | A picture of what the data looks like |
| **Code** | A query you run yourself |
| **Expected output** | What you should see — check your result against it |
| **Exercises** | 2–5 problems, with collapsible solutions |
| **Checkpoint** | An automated self-test. Don't move on until it passes |

> **Rule of the course:** Never copy a query without predicting its output first. Guess, then run, then compare. That gap between your guess and reality is where the learning happens.

---

### Course map

| Part | Title | You will be able to... |
|---|---|---|
| 0 | Setup & Connection | Connect Python to Neo4j Aura |
| 1 | Why Graph Databases? | Explain *when* a graph beats a relational DB — with measurements |
| 2 | The Property Graph Model | Model a domain as nodes, relationships, labels, properties |
| 3 | Loading Data | Load 12k nodes / 37k relationships safely and fast |
| 4 | Cypher Fundamentals | `MATCH`, `WHERE`, `RETURN`, `ORDER BY`, `LIMIT` |
| 5 | Aggregation & Pipelines | `count`, `collect`, `WITH`, implicit grouping |
| 6 | Traversal & Paths | Multi-hop, variable-length, shortest path |
| 7 | Advanced Cypher | `OPTIONAL MATCH`, subqueries, comprehensions, `CASE` |
| 8 | Graph Analytics | Recommendation engines & anomaly detection in pure Cypher |
| 9 | Performance | Indexes, `PROFILE`, query plans, anti-patterns |
| 10 | Writing & Refactoring | `CREATE`, `MERGE`, `SET`, `DELETE`, schema evolution |
| 11 | Modelling Deep Dive | Choose between competing models and justify it |
| 12 | Capstone | Solve an open-ended business problem end-to-end |

**Estimated time:** 8–12 hours of class contact, or 3 sessions of ~4 hours.

---
# Part 0 — Setup & Connection

## 0.1 What you need

1. **Python 3.9+** with Jupyter (you already have this — you're reading it).
2. **A free Neo4j Aura instance** (cloud, no install, no credit card).
3. **The dataset folder** `ecommerce_graph_data/` sitting next to this notebook.

Nothing gets installed on your machine except three Python packages.

In [ ]:
# Install the Python packages this course needs.
# %pip works inside Jupyter and installs into the *kernel's* environment.
%pip install --quiet "neo4j>=5.14" pandas matplotlib networkx
print("Packages installed.")

## 0.2 Create your free Neo4j Aura instance

Neo4j Aura Free gives you a real, cloud-hosted graph database — permanently free, capped at
**200,000 nodes** and **400,000 relationships**. Our dataset uses about 6% of that, so you have
plenty of room to experiment.

**Follow these steps exactly:**

1. Go to **<https://console.neo4j.io>** and sign up (Google/GitHub sign-in is fastest).
2. Click **New Instance** → choose **AuraDB Free** → **Create**.
3. A panel appears showing your **username** (`neo4j`) and a **generated password**.
   > ⚠️ **Download or copy that password now.** Neo4j shows it exactly once and cannot recover it.
   > If you lose it you must reset the instance.
4. Wait ~60 seconds until the instance status turns from *Creating* to **Running**.
5. Copy the **Connection URI**. It looks like:
   ```
   neo4j+s://a1b2c3d4.databases.neo4j.io
   ```

### What do those pieces mean?

| Piece | Meaning |
|---|---|
| `neo4j+s://` | The Bolt protocol over TLS (`+s` = encrypted, certificate verified). Aura *requires* this. |
| `a1b2c3d4` | Your instance ID |
| `:7687` | The Bolt port — implied, you don't type it |

**Bolt** is Neo4j's binary protocol, purpose-built for graph traffic. It streams records back as
you consume them rather than materialising the whole result — which is why you can ask for a
million rows without exhausting memory.

In [ ]:
# Store your credentials. getpass hides the password so it never gets saved
# into the notebook file — important if you share or commit this notebook.
from getpass import getpass

NEO4J_URI      = input("Connection URI (neo4j+s://xxxx.databases.neo4j.io): ").strip()
NEO4J_USER     = input("Username [neo4j]: ").strip() or "neo4j"
NEO4J_PASSWORD = getpass("Password: ")

print(f"\nURI set to: {NEO4J_URI}")
print(f"User:       {NEO4J_USER}")
print(f"Password:   {'*' * len(NEO4J_PASSWORD)}  ({len(NEO4J_PASSWORD)} chars)")

## 0.3 The connection helper

Rather than repeating boilerplate in every cell, we build one small helper class.
Read it carefully — you will use `gdb.run(...)` hundreds of times today.

**Key design decisions, and why:**

- **One `Driver` for the whole notebook.** The driver holds a *connection pool*. Creating a
  driver per query would open a new TCP+TLS connection every time — slow and wasteful.
  Create it once, reuse it, close it at the end.
- **`session()` per query.** A session is a cheap, short-lived context that borrows a connection
  from the pool and guarantees *causal consistency* (your reads see your own writes).
- **Results come back as a pandas DataFrame** so they render as tidy tables in Jupyter.
- **Parameters, never string concatenation.** `$name` placeholders let Neo4j cache the query
  plan and make injection attacks impossible.

In [ ]:
import time
import pandas as pd
from neo4j import GraphDatabase
from neo4j.exceptions import Neo4jError

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


class GraphDB:
    """A thin, teaching-friendly wrapper around the Neo4j Python driver."""

    def __init__(self, uri, user, password, database="neo4j"):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        self.database = database
        self.driver.verify_connectivity()   # fails fast with a clear error

    def run(self, cypher, params=None, show_time=False):
        """Run a query and return the results as a pandas DataFrame."""
        started = time.perf_counter()
        try:
            with self.driver.session(database=self.database) as session:
                result = session.run(cypher, params or {})
                df = result.to_df()
                summary = result.consume()
        except Neo4jError as e:
            # Neo4j's error messages are excellent - surface them clearly.
            print(f"CYPHER ERROR [{e.code}]\n{e.message}")
            raise
        elapsed = (time.perf_counter() - started) * 1000
        if show_time:
            counters = summary.counters
            changes = []
            for field in ("nodes_created", "nodes_deleted", "relationships_created",
                          "relationships_deleted", "properties_set", "labels_added"):
                value = getattr(counters, field, 0)
                if value:
                    changes.append(f"{field}={value}")
            note = "  " + ", ".join(changes) if changes else ""
            print(f"[{elapsed:7.1f} ms  {len(df):>6,} rows]{note}")
        return df

    def graph(self, cypher, params=None):
        """Return the raw node/relationship graph objects (used for drawing)."""
        with self.driver.session(database=self.database) as session:
            return session.run(cypher, params or {}).graph()

    def close(self):
        self.driver.close()


gdb = GraphDB(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
print("Connected to Neo4j.")

In [ ]:
# Confirm which Neo4j version and edition you are talking to.
gdb.run("CALL dbms.components() YIELD name, versions, edition "
        "RETURN name, versions[0] AS version, edition")

**Expected output:** one row — `Neo4j Kernel`, a version like `5.x.x`, edition `enterprise`
(Aura runs Enterprise under the hood even on the Free tier).

## 0.4 A drawing helper

Neo4j's own Browser draws beautiful graphs, but we want pictures *inside the notebook*.
This helper converts any query that returns nodes and relationships into a `networkx` drawing.

You do not need to understand this code to do the course — treat it as a black box for now
and come back to it later if you're curious.

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

# A consistent colour per node label, so diagrams read the same all course long.
LABEL_COLOURS = {
    "Customer": "#4C9BE8", "Product": "#F2A65A", "Order": "#7FBF7F",
    "Category": "#B98FD9", "Supplier": "#E86A6A", "Warehouse": "#5FC9C9",
    "Review": "#F2D06B", "Brand": "#C98FB5", "Tag": "#9FB8AD",
}

# Which property to show as the caption for each label.
LABEL_CAPTION = {
    "Customer": "name", "Product": "name", "Order": "orderId",
    "Category": "name", "Supplier": "name", "Warehouse": "name",
    "Review": "rating", "Brand": "name", "Tag": "name",
}


def draw(cypher, params=None, title="", figsize=(13, 8), seed=7, font_size=8):
    """Run a query that returns nodes/relationships and draw the result."""
    g = gdb.graph(cypher, params)
    if len(g.nodes) == 0:
        print("Query returned no nodes to draw.")
        return

    G = nx.MultiDiGraph()
    colours, captions = [], {}
    for node in g.nodes:
        label = next(iter(node.labels), "Node")
        prop = LABEL_CAPTION.get(label, "name")
        caption = node.get(prop, node.get("name", label))
        if isinstance(caption, str) and len(caption) > 22:
            caption = caption[:20] + ".."
        G.add_node(node.element_id)
        colours.append(LABEL_COLOURS.get(label, "#CCCCCC"))
        captions[node.element_id] = f"{label}\n{caption}"

    edge_labels = {}
    for rel in g.relationships:
        s, e = rel.start_node.element_id, rel.end_node.element_id
        G.add_edge(s, e)
        edge_labels[(s, e)] = rel.type

    pos = nx.spring_layout(G, seed=seed, k=1.1 / max(len(G) ** 0.5, 1))
    plt.figure(figsize=figsize)
    nx.draw_networkx_nodes(G, pos, node_color=colours, node_size=2100,
                           edgecolors="#333333", linewidths=1.2)
    nx.draw_networkx_edges(G, pos, edge_color="#888888", arrows=True,
                           arrowsize=16, node_size=2100, width=1.3,
                           connectionstyle="arc3,rad=0.06")
    nx.draw_networkx_labels(G, pos, captions, font_size=font_size)
    nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=font_size - 1,
                                 font_color="#555555", rotate=False)
    plt.title(title or f"{len(g.nodes)} nodes, {len(g.relationships)} relationships",
              fontsize=12)
    plt.axis("off")
    plt.tight_layout()
    plt.show()


def legend():
    """Show the node colour key."""
    fig, ax = plt.subplots(figsize=(10, 1.1))
    for i, (label, colour) in enumerate(LABEL_COLOURS.items()):
        ax.scatter(i, 0, s=420, c=colour, edgecolors="#333")
        ax.text(i, -0.38, label, ha="center", fontsize=9)
    ax.set_xlim(-0.6, len(LABEL_COLOURS) - 0.4)
    ax.set_ylim(-0.8, 0.5)
    ax.axis("off")
    plt.title("Node colour key", fontsize=10)
    plt.show()


legend()
print("Drawing helpers ready.")

## 0.5 Checkpoint 0

Run the cell below. All four checks must print `PASS` before you continue.

In [ ]:
def checkpoint(name, checks):
    """checks = list of (description, boolean) tuples."""
    print(f"{'=' * 62}\nCHECKPOINT: {name}\n{'=' * 62}")
    passed = 0
    for desc, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}]  {desc}")
        passed += bool(ok)
    print(f"{'-' * 62}\n  {passed}/{len(checks)} passed"
          f"{'  — you may continue.' if passed == len(checks) else '  — fix the failures above.'}\n")
    return passed == len(checks)


import pathlib
checkpoint("Part 0 — Environment", [
    ("Driver connected",         gdb.run("RETURN 1 AS ok").iloc[0]["ok"] == 1),
    ("Can read server version",  len(gdb.run("CALL dbms.components()")) > 0),
    ("Dataset folder present",   pathlib.Path("ecommerce_graph_data").is_dir()),
    ("customers.csv found",      pathlib.Path("ecommerce_graph_data/customers.csv").is_file()),
])

---
# Part 1 — Why Graph Databases?

> **Do not skip this part.** Anyone can learn Cypher syntax in an afternoon. Knowing *when to
> reach for a graph database* is the skill that makes you valuable — and it's the thing
> interviewers actually probe.

## 1.1 The claim we're going to test

Here is the claim, stated plainly:

> **Relational databases store entities well and relationships poorly.
> Graph databases store relationships as first-class citizens.**

That sounds like marketing. So we're not going to take it on faith — we're going to
**build the same dataset twice**, once relationally (in SQLite) and once as a graph, and
**measure** the difference on the same questions.

## 1.2 The business question

Our e-commerce company wants a recommendation feature:

> *"Customers who bought this product also bought..."*

To answer it you must walk a chain:

```
Product  ←  OrderItem  ←  Order  →  Customer  →  Order  →  OrderItem  →  Product
```

That's **six hops**. Let's see what each technology has to do to answer it.

## 1.3 Build the relational version

We'll load the raw CSVs into SQLite — a genuine relational database with a real query planner
and real B-tree indexes. This is a fair fight.

In [ ]:
import sqlite3, pathlib, time
import pandas as pd

DATA = pathlib.Path("ecommerce_graph_data")
con = sqlite3.connect(":memory:")   # in-memory = as fast as SQL can possibly be

TABLES = ["customers", "products", "categories", "orders", "order_items",
          "reviews", "suppliers", "product_suppliers", "inventory",
          "warehouses", "shipments", "payment_transactions"]

for t in TABLES:
    pd.read_csv(DATA / f"{t}.csv").to_sql(t, con, index=False, if_exists="replace")

# Give SQLite every advantage: index every join key.
for stmt in [
    "CREATE INDEX ix_o_cust  ON orders(customer_id)",
    "CREATE INDEX ix_o_id    ON orders(order_id)",
    "CREATE INDEX ix_oi_ord  ON order_items(order_id)",
    "CREATE INDEX ix_oi_prod ON order_items(product_id)",
    "CREATE INDEX ix_c_id    ON customers(customer_id)",
    "CREATE INDEX ix_p_id    ON products(product_id)",
    "CREATE INDEX ix_r_prod  ON reviews(product_id)",
    "CREATE INDEX ix_r_cust  ON reviews(customer_id)",
]:
    con.execute(stmt)
con.commit()

sql = lambda q, p=(): pd.read_sql(q, con, params=p)

print("SQLite loaded and fully indexed:")
for t in TABLES:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"   {t:22s} {n:>7,} rows")

## 1.4 Watch the JOINs multiply

Let's ask progressively deeper questions and count the JOINs each one needs.
Pay attention to how the SQL grows — that growth *is* the lesson.

In [ ]:
# HOP 1 — "What did customer 1001 order?"  (1 join)
q1 = """
SELECT o.order_id, o.order_date, o.total_amount
FROM   orders o
WHERE  o.customer_id = ?
LIMIT 5
"""

# HOP 2 — "Which products did they buy?"   (2 joins)
q2 = """
SELECT p.product_name, oi.quantity
FROM   orders o
JOIN   order_items oi ON oi.order_id   = o.order_id
JOIN   products    p  ON p.product_id  = oi.product_id
WHERE  o.customer_id = ?
LIMIT 5
"""

# HOP 4 — "Who else bought those products?" (4 joins)
q4 = """
SELECT DISTINCT c2.first_name, c2.last_name
FROM   orders o1
JOIN   order_items oi1 ON oi1.order_id  = o1.order_id
JOIN   order_items oi2 ON oi2.product_id = oi1.product_id
JOIN   orders      o2  ON o2.order_id   = oi2.order_id
JOIN   customers   c2  ON c2.customer_id = o2.customer_id
WHERE  o1.customer_id = ? AND c2.customer_id <> ?
LIMIT 5
"""

# HOP 6 — "...and what ELSE did those people buy?"  (6 joins) — the recommendation
q6 = """
SELECT   p2.product_name, COUNT(DISTINCT o2.customer_id) AS shoppers
FROM     orders o1
JOIN     order_items oi1 ON oi1.order_id   = o1.order_id
JOIN     order_items oi2 ON oi2.product_id = oi1.product_id
JOIN     orders      o2  ON o2.order_id    = oi2.order_id
JOIN     order_items oi3 ON oi3.order_id   = o2.order_id
JOIN     products    p2  ON p2.product_id  = oi3.product_id
WHERE    o1.customer_id = ?
  AND    o2.customer_id <> ?
  AND    p2.product_id NOT IN (
            SELECT oi.product_id FROM orders o
            JOIN order_items oi ON oi.order_id = o.order_id
            WHERE o.customer_id = ?)
GROUP BY p2.product_id
ORDER BY shoppers DESC
LIMIT 5
"""

# Pick a customer who actually has orders, so every query returns rows.
CID = int(sql("SELECT customer_id FROM orders GROUP BY customer_id "
              "ORDER BY COUNT(*) DESC LIMIT 1").iloc[0, 0])
print(f"Using customer_id = {CID}\n")

for label, q, params in [("1 hop  (1 join) ", q1, (CID,)),
                         ("2 hops (2 joins)", q2, (CID,)),
                         ("4 hops (4 joins)", q4, (CID, CID)),
                         ("6 hops (6 joins)", q6, (CID, CID, CID))]:
    t0 = time.perf_counter()
    rows = sql(q, params)
    ms = (time.perf_counter() - t0) * 1000
    print(f"{label}   {ms:7.2f} ms   {len(rows)} rows")

print("\nThe 6-hop recommendation result:")
sql(q6, (CID, CID, CID))

### Read the SQL again, and notice three things

1. **The query text grew roughly linearly with depth** — 1 join, 2 joins, 4 joins, 6 joins.
   A "friends-of-friends-of-friends" question in a social graph would need 3 self-joins; ten
   hops would be unwritable by hand.

2. **You had to name the join keys every single time.** `oi.order_id = o.order_id`,
   `p.product_id = oi.product_id`... The database does not *know* that orders contain items.
   You re-teach it that fact in every query.

3. **Each join is an index lookup.** To follow `order → order_items`, SQLite searches a B-tree
   on `order_items(order_id)`. That's `O(log n)` per hop — cheap, but it happens for *every row
   at every level*, and the intermediate result set can explode before the final filter shrinks it.

> **This is the crux.** In a relational database, a relationship is *computed at query time*
> by matching values through an index. It is not stored.

## 1.5 How a graph database does it instead: index-free adjacency

In Neo4j, when you create a relationship between two nodes, the database writes a
**physical pointer** into the record of each node.

Following a relationship is then a **pointer dereference** — the same operation as following a
pointer in C, or a `next` pointer in a linked list. It costs the same whether your database
holds a thousand nodes or a billion.

This property is called **index-free adjacency**, and it is *the* defining feature of a native
graph database.

<div align="center">

<svg width="820" height="360" xmlns="http://www.w3.org/2000/svg" style="background:#fff">
  <style>
    .t{font-family:-apple-system,Segoe UI,sans-serif;font-size:13px;fill:#222}
    .tb{font-family:-apple-system,Segoe UI,sans-serif;font-size:14px;font-weight:700;fill:#111}
    .ts{font-family:-apple-system,Segoe UI,sans-serif;font-size:11px;fill:#666}
    .box{fill:#fff;stroke:#555;stroke-width:1.4}
    .idx{fill:#FDE8D0;stroke:#D08A3E;stroke-width:1.4}
    .nd{fill:#4C9BE8;stroke:#2E6DA4;stroke-width:1.6}
    .arw{stroke:#888;stroke-width:1.6;fill:none;marker-end:url(#a)}
    .arw2{stroke:#2E8B57;stroke-width:2.4;fill:none;marker-end:url(#b)}
  </style>
  <defs>
    <marker id="a" markerWidth="9" markerHeight="9" refX="8" refY="4.5" orient="auto">
      <path d="M0,0 L9,4.5 L0,9 z" fill="#888"/></marker>
    <marker id="b" markerWidth="9" markerHeight="9" refX="8" refY="4.5" orient="auto">
      <path d="M0,0 L9,4.5 L0,9 z" fill="#2E8B57"/></marker>
  </defs>

  <text x="15" y="24" class="tb">RELATIONAL — every hop is an index lookup</text>
  <rect x="15" y="40" width="120" height="46" class="box" rx="4"/>
  <text x="30" y="62" class="t">orders row</text><text x="30" y="78" class="ts">id = 3001</text>
  <path d="M140,63 L215,63" class="arw"/>
  <rect x="220" y="34" width="150" height="58" class="idx" rx="4"/>
  <text x="235" y="55" class="t">B-tree index on</text>
  <text x="235" y="72" class="ts">order_items(order_id)</text>
  <text x="235" y="86" class="ts">O(log n) search</text>
  <path d="M375,63 L450,63" class="arw"/>
  <rect x="455" y="40" width="130" height="46" class="box" rx="4"/>
  <text x="470" y="62" class="t">order_items row</text><text x="470" y="78" class="ts">product_id = 2169</text>
  <path d="M590,63 L650,63" class="arw"/>
  <rect x="655" y="34" width="145" height="58" class="idx" rx="4"/>
  <text x="668" y="55" class="t">B-tree index on</text>
  <text x="668" y="72" class="ts">products(product_id)</text>
  <text x="668" y="86" class="ts">O(log n) search</text>
  <text x="15" y="118" class="ts">Cost grows with total table size — and repeats for every intermediate row.</text>

  <line x1="15" y1="140" x2="800" y2="140" stroke="#ddd" stroke-width="1"/>

  <text x="15" y="172" class="tb">GRAPH — every hop is a pointer dereference</text>
  <circle cx="80" cy="240" r="34" class="nd"/>
  <text x="80" y="238" class="t" fill="#fff" text-anchor="middle" font-weight="700">Order</text>
  <text x="80" y="252" class="ts" fill="#eaf2fb" text-anchor="middle">3001</text>
  <path d="M118,240 L232,240" class="arw2"/>
  <text x="175" y="230" class="ts" text-anchor="middle" fill="#2E8B57">CONTAINS</text>
  <text x="175" y="256" class="ts" text-anchor="middle">stored pointer</text>
  <circle cx="270" cy="240" r="34" style="fill:#F2A65A;stroke:#B87333;stroke-width:1.6"/>
  <text x="270" y="238" class="t" fill="#fff" text-anchor="middle" font-weight="700">Product</text>
  <text x="270" y="252" class="ts" fill="#fff8f0" text-anchor="middle">2169</text>
  <path d="M308,240 L422,240" class="arw2"/>
  <text x="365" y="230" class="ts" text-anchor="middle" fill="#2E8B57">IN_CATEGORY</text>
  <text x="365" y="256" class="ts" text-anchor="middle">stored pointer</text>
  <circle cx="460" cy="240" r="34" style="fill:#B98FD9;stroke:#8B5FB0;stroke-width:1.6"/>
  <text x="460" y="238" class="t" fill="#fff" text-anchor="middle" font-weight="700">Category</text>
  <text x="460" y="252" class="ts" fill="#f8f0ff" text-anchor="middle">Shoes</text>
  <path d="M498,240 L612,240" class="arw2"/>
  <text x="555" y="230" class="ts" text-anchor="middle" fill="#2E8B57">PARENT_OF</text>
  <circle cx="650" cy="240" r="34" style="fill:#B98FD9;stroke:#8B5FB0;stroke-width:1.6"/>
  <text x="650" y="244" class="t" fill="#fff" text-anchor="middle" font-weight="700">Clothing</text>
  <text x="15" y="312" class="ts">Cost is O(1) per hop and is completely independent of how big the database is.</text>
  <text x="15" y="332" class="ts">A 10-hop query on 10 nodes costs the same per hop as a 10-hop query on 10 billion nodes.</text>
</svg>

</div>

## 1.6 The same recommendation, in Cypher

We haven't loaded the graph yet (that's Part 3), so read this now and *run* it later.
Here is the exact same six-hop recommendation:

```cypher
MATCH (me:Customer {customerId: $cid})-[:PLACED]->(:Order)-[:CONTAINS]->(bought:Product)
MATCH (bought)<-[:CONTAINS]-(:Order)<-[:PLACED]-(peer:Customer)
WHERE peer <> me
MATCH (peer)-[:PLACED]->(:Order)-[:CONTAINS]->(rec:Product)
WHERE NOT (me)-[:PLACED]->(:Order)-[:CONTAINS]->(rec)
RETURN rec.name AS recommendation, count(DISTINCT peer) AS shoppers
ORDER BY shoppers DESC LIMIT 5
```

Compare the two side by side:

| | SQL | Cypher |
|---|---|---|
| Lines | ~18 | 7 |
| Join conditions you must write | 6 | 0 |
| Reads like the question? | No | Yes |
| Cost per hop | `O(log n)` index seek | `O(1)` pointer hop |
| Cost as data grows 100× | Grows | Unchanged |

The Cypher version has **zero join predicates**. You never write `oi.order_id = o.order_id`,
because the relationship `CONTAINS` *is* that fact, stored once at write time instead of being
re-derived on every read.

And notice the **ASCII-art pattern**: `(a)-[:REL]->(b)` literally draws the shape you're looking
for. That's not a gimmick — it's why graph queries stay readable at depth where SQL doesn't.

## 1.7 The honest counter-argument: when NOT to use a graph

A good engineer knows the limits of their tool. **Graph databases are a poor choice when:**

| Situation | Why a graph is wrong | Use instead |
|---|---|---|
| Bulk aggregation over one huge table<br>*("sum revenue by month for 5 years")* | You're scanning, not traversing. Relationships add overhead with no benefit. | Columnar store — Snowflake, BigQuery, DuckDB, Databricks |
| Rigid, tabular, unchanging schema<br>*(ledger, payroll)* | Relational constraints and ACID reporting are more mature | PostgreSQL |
| Simple key → value lookup | A graph is enormous overkill | Redis, DynamoDB |
| Full-text search over documents | Graphs index properties, not language | Elasticsearch, OpenSearch |
| Append-only time-series metrics | No meaningful relationships between points | TimescaleDB, InfluxDB, Prometheus |

**Reach for a graph when the *relationships themselves* carry the value:**

- **Recommendations** — "people like you also bought"
- **Fraud rings** — accounts sharing a device, address, or card
- **Network & IT topology** — blast-radius analysis, dependency chains
- **Identity & access** — who can reach what, through which group, transitively
- **Knowledge graphs & RAG** — entities linked by typed, meaningful edges
- **Supply chains** — trace a contaminated batch back through every supplier tier
- **Master data management** — resolving that six customer records are one person

> **The heuristic:** if your SQL has more than three self-joins, or a recursive CTE, or you find
> yourself writing `WITH RECURSIVE` — you have a graph problem wearing a relational costume.

## 1.8 Exercises — Part 1

Answer in a markdown cell in your own words. These are discussion questions; there's a
suggested answer under each.

**Exercise 1.1** — Write the SQL for *"find all customers within 3 degrees of customer 1001,
connected through shared purchased products."* Don't run it — just write it, and count the JOINs.

<details>
<summary><b>&#128161; Show solution &mdash; 1.1</b></summary>


You would need roughly this, and it gets worse at each degree:

```sql
SELECT DISTINCT c3.customer_id
FROM orders o1
JOIN order_items oi1 ON oi1.order_id = o1.order_id
JOIN order_items oi2 ON oi2.product_id = oi1.product_id
JOIN orders o2 ON o2.order_id = oi2.order_id            -- degree 1
JOIN order_items oi3 ON oi3.order_id = o2.order_id
JOIN order_items oi4 ON oi4.product_id = oi3.product_id
JOIN orders o3 ON o3.order_id = oi4.order_id            -- degree 2
JOIN order_items oi5 ON oi5.order_id = o3.order_id
JOIN order_items oi6 ON oi6.product_id = oi5.product_id
JOIN orders o4 ON o4.order_id = oi6.order_id
JOIN customers c3 ON c3.customer_id = o4.customer_id    -- degree 3
WHERE o1.customer_id = 1001;
```

**11 JOINs**, and the query must be *rewritten* to change the depth.

The Cypher equivalent is one line, and the depth is a parameter:

```cypher
MATCH (me:Customer {customerId:1001})
      -[:PLACED|CONTAINS*1..6]-(other:Customer)
RETURN DISTINCT other
```


</details>

**Exercise 1.2** — Your company stores IoT sensor readings: 50 million rows per day, each row
is `(sensor_id, timestamp, value)`. Queries are always *"average value for sensor X over the
last hour."* Is a graph database a good fit? Justify your answer.

<details>
<summary><b>&#128161; Show solution &mdash; 1.2</b></summary>


**No.** There are essentially no relationships being traversed — each reading connects to
exactly one sensor, and queries are range scans over time plus an aggregation. That is a
time-series/columnar workload.

Index-free adjacency buys you nothing when you never traverse. You'd pay graph storage overhead
and get worse scan performance than TimescaleDB, InfluxDB or a columnar store.

**Nuance worth raising in class:** a graph *could* be useful for the sensor *topology* —
which sensor feeds which controller, which controller is on which production line, so you can
answer *"if sensor X fails, what downstream systems are affected?"* That's a genuine graph
question. The common architecture is **both**: readings in a time-series store, topology in a graph.


</details>

**Exercise 1.3** — Explain index-free adjacency to a non-technical manager in three sentences,
without using the words "index", "pointer", or "join".

<details>
<summary><b>&#128161; Show solution &mdash; 1.3</b></summary>


One good answer:

> "In a normal database, finding what's connected to something is like looking up a name in a
> phone book every single time — and the book gets slower to search as it gets thicker.
> In a graph database, each record keeps a direct link to the things it's connected to, like a
> contact list already saved on your phone.
> So following connections stays instant no matter how much data we add."

Marking guidance: the key idea students must convey is *the cost of following a connection
doesn't grow with the size of the database*.


</details>

## 1.9 Checkpoint 1

In [ ]:
recs = sql(q6, (CID, CID, CID))
checkpoint("Part 1 — Relational baseline", [
    ("SQLite loaded with 12 tables",
     len(sql("SELECT name FROM sqlite_master WHERE type='table'")) >= 12),
    ("6-hop SQL recommendation returned rows", len(recs) > 0),
    ("Orders table has > 5,000 rows",
     sql("SELECT COUNT(*) c FROM orders").iloc[0]["c"] > 5000),
    ("You can state one case where a graph DB is the WRONG choice", True),  # honour system
])

---
# Part 2 — The Property Graph Model

Neo4j implements the **labelled property graph** model. It has exactly four building blocks.
Learn these four and you understand the entire data model.

## 2.1 The four building blocks

<div align="center">

<svg width="800" height="330" xmlns="http://www.w3.org/2000/svg" style="background:#fff">
  <style>
    .t{font-family:-apple-system,Segoe UI,sans-serif;font-size:13px;fill:#222}
    .tb{font-family:-apple-system,Segoe UI,sans-serif;font-size:15px;font-weight:700;fill:#111}
    .ts{font-family:-apple-system,Segoe UI,sans-serif;font-size:11px;fill:#666}
    .cal{font-family:-apple-system,Segoe UI,sans-serif;font-size:12px;font-weight:700;fill:#C0392B}
  </style>
  <defs>
    <marker id="ar" markerWidth="10" markerHeight="10" refX="9" refY="5" orient="auto">
      <path d="M0,0 L10,5 L0,10 z" fill="#444"/></marker>
  </defs>

  <circle cx="180" cy="130" r="62" fill="#4C9BE8" stroke="#2E6DA4" stroke-width="2"/>
  <text x="180" y="122" text-anchor="middle" class="tb" fill="#fff">Customer</text>
  <text x="180" y="142" text-anchor="middle" class="ts" fill="#eaf2fb">Stacy Nielsen</text>

  <circle cx="600" cy="130" r="62" fill="#F2A65A" stroke="#B87333" stroke-width="2"/>
  <text x="600" y="122" text-anchor="middle" class="tb" fill="#fff">Product</text>
  <text x="600" y="142" text-anchor="middle" class="ts" fill="#fff8f0">Smart Shirt</text>

  <path d="M244,130 L534,130" stroke="#444" stroke-width="2.4" marker-end="url(#ar)"/>
  <rect x="330" y="106" width="120" height="24" fill="#fff" stroke="#444" rx="12"/>
  <text x="390" y="123" text-anchor="middle" class="t" font-weight="700">REVIEWED</text>
  <text x="390" y="152" text-anchor="middle" class="ts">rating: 5</text>
  <text x="390" y="167" text-anchor="middle" class="ts">date: 2023-10-27</text>

  <text x="60" y="42" class="cal">1. NODE</text>
  <text x="60" y="60" class="ts">an entity — a thing</text>
  <path d="M95,66 L150,80" stroke="#C0392B" stroke-width="1.2" fill="none"/>

  <text x="250" y="252" class="cal">2. LABEL</text>
  <text x="250" y="270" class="ts">the node's type / category</text>
  <path d="M262,244 L205,180" stroke="#C0392B" stroke-width="1.2" fill="none"/>

  <text x="330" y="42" class="cal">3. RELATIONSHIP</text>
  <text x="330" y="60" class="ts">typed AND directed — always</text>
  <path d="M390,66 L390,100" stroke="#C0392B" stroke-width="1.2" fill="none"/>

  <text x="560" y="252" class="cal">4. PROPERTIES</text>
  <text x="500" y="270" class="ts">key/value pairs — on nodes AND on relationships</text>
  <path d="M580,244 L440,175" stroke="#C0392B" stroke-width="1.2" fill="none"/>
  <path d="M600,244 L600,196" stroke="#C0392B" stroke-width="1.2" fill="none"/>

  <text x="40" y="312" class="ts">Cypher for exactly this picture:  (c:Customer {name:'Stacy Nielsen'})-[r:REVIEWED {rating:5}]-&gt;(p:Product {name:'Smart Shirt'})</text>
</svg>

</div>

### 1. Node — a *thing*

An entity: a customer, a product, an order. Drawn as a circle. Written as `()` in Cypher.

### 2. Label — a node's *type*

`:Customer`, `:Product`. Labels group nodes so Neo4j can find them fast — an index is scoped
to a label. A node may have **zero, one, or many** labels:

```cypher
(:Customer:VIP:EmailSubscriber)   // all three are true of this node
```

Multiple labels are how you model *roles and states* without extra columns. A node can gain and
lose a label at runtime (`SET n:VIP`, `REMOVE n:VIP`).

### 3. Relationship — a *typed, directed* connection

Written `-[:BOUGHT]->`. Every relationship in Neo4j:

- has **exactly one type** (unlike labels, never zero or many)
- has **exactly one direction** (always stored with a start and end node)
- connects **exactly two nodes** (which may be the same node — self-loops are legal)

> **Crucial point students always trip on:** direction is *stored* but need not be *queried*.
> `(a)-[:KNOWS]-(b)` with no arrow matches the relationship in either direction, at no extra cost.
> So model direction the way it naturally reads (`Customer -[:PLACED]-> Order`) and query it
> whichever way you need. **Never create two relationships just to represent "both directions".**

### 4. Property — a key/value pair

Stored on nodes *and* on relationships. Supported types: string, integer, float, boolean,
point (spatial), the temporal types (date, datetime, duration...), and homogeneous lists of those.

> **There is no nesting.** A property value cannot be a map or an object. If you find yourself
> wanting `customer.address.city`, that's the model telling you `Address` should be its own node.

## 2.2 The modelling decision that matters most

Beginners ask: *"should this be a property, or a node?"* Here is the rule that resolves 90% of cases.

> ### Make it a NODE if you ever want to ask a question *about it*, or *traverse through it*.
> ### Leave it a PROPERTY if it's only ever an attribute you filter or display.

Worked examples from our dataset:

| Thing | Decision | Why |
|---|---|---|
| `customer.email` | **Property** | You filter and display it. You never ask "what else connects to this email?" |
| `product.price` | **Property** | Pure attribute. Filter, sort, aggregate. |
| Product **category** | **NODE** | Categories form a hierarchy. You traverse up and down them. |
| Product **brand** | **Node** (eventually) | "Which brands does this customer prefer?" traverses *through* brand |
| Order **status** | **Property** | Just a filter value... |
| Order **status** | ...**or a label** | `:Order:Cancelled` if you filter on it constantly — labels are indexed |
| `order_items` table | **RELATIONSHIP** with properties | It's a pure join table. `quantity` and `unitPrice` live on the edge. |
| **Address** | Depends! | Property if display-only. **Node** if you hunt fraud rings sharing an address. |

That last row is the deepest lesson in graph modelling:

> **The right model depends on the questions you intend to ask.**
> There is no single "correct" schema for a domain — only a schema that is correct *for a workload*.
> This is genuinely different from relational modelling, where normalisation gives you a
> question-independent answer.

## 2.3 Translating relational → graph

Here's the mechanical translation you can apply to any relational schema:

| Relational | Graph | Note |
|---|---|---|
| Table row | Node | |
| Table name | Label | |
| Column | Property | |
| Primary key | Property + **uniqueness constraint** | Neo4j has its own internal IDs — never rely on them |
| Foreign key | **Relationship** | The FK column itself usually disappears |
| Join table (2 FKs) | **Relationship** | Extra columns become relationship properties |
| Join table (3+ FKs) | **Intermediate node** | A relationship connects only 2 nodes, so you need a hub |
| `NULL` column | **Absent property** | Graphs are schema-optional — no space wasted on nulls |
| Lookup/dimension table | Node (often small, high-degree) | |

Applying this to our e-commerce schema:

- `customers` → `(:Customer)` nodes
- `orders` → `(:Order)` nodes, and `orders.customer_id` becomes `(:Customer)-[:PLACED]->(:Order)`
- `order_items` → the relationship `(:Order)-[:CONTAINS {quantity, unitPrice}]->(:Product)`
- `products.category` → `(:Product)-[:IN_CATEGORY]->(:Category)`
- `categories.parent_category_id` → `(:Category)-[:PARENT_OF]->(:Category)` — a *self*-relationship
- `product_suppliers` → `(:Supplier)-[:SUPPLIES {costPrice, leadTimeDays}]->(:Product)`
- `inventory` → `(:Warehouse)-[:STOCKS {quantity}]->(:Product)`
- `reviews` → **a node**, not a relationship. Why? See below.

### Why is Review a node when order_items is a relationship?

Both are "many-to-many between two entities". The difference:

- `order_items` carries only *attributes of the connection* (quantity, price). Nothing else ever
  connects to an order line.
- A **review** is a thing in its own right. It has text, votes, a sentiment, a date; it can be
  responded to, flagged, moderated, or replied to. Other things will eventually point *at it*.

> **Rule:** if the connection needs its own relationships, it must be a node.
> A relationship can't connect to another relationship.

## 2.4 Our target graph model

This is the schema we'll build in Part 3. Study it before you load it.

<div align="center">

<svg width="860" height="520" xmlns="http://www.w3.org/2000/svg" style="background:#fff">
  <style>
    .lb{font-family:-apple-system,Segoe UI,sans-serif;font-size:13px;font-weight:700;fill:#fff;text-anchor:middle}
    .rl{font-family:-apple-system,Segoe UI,sans-serif;font-size:11px;font-weight:700;fill:#2c3e50;text-anchor:middle}
    .pp{font-family:-apple-system,Segoe UI,sans-serif;font-size:10px;fill:#7f8c8d;text-anchor:middle}
  </style>
  <defs>
    <marker id="m" markerWidth="9" markerHeight="9" refX="8" refY="4.5" orient="auto">
      <path d="M0,0 L9,4.5 L0,9 z" fill="#7f8c8d"/></marker>
  </defs>

  <!-- edges first so they sit under the nodes -->
  <path d="M150,120 L150,196" stroke="#7f8c8d" stroke-width="2" marker-end="url(#m)"/>
  <text x="112" y="162" class="rl">PLACED</text>

  <path d="M204,232 L386,232" stroke="#7f8c8d" stroke-width="2" marker-end="url(#m)"/>
  <text x="295" y="222" class="rl">CONTAINS</text>
  <text x="295" y="246" class="pp">quantity, unitPrice, totalPrice</text>

  <path d="M440,196 L440,120" stroke="#7f8c8d" stroke-width="2" marker-end="url(#m)"/>
  <text x="392" y="162" class="rl">IN_CATEGORY</text>

  <path d="M488,84 C560,60 588,60 620,84" stroke="#7f8c8d" stroke-width="2" marker-end="url(#m)"/>
  <text x="556" y="52" class="rl">PARENT_OF</text>
  <text x="556" y="36" class="pp">(hierarchy, 1–3 levels)</text>

  <path d="M150,268 L150,352" stroke="#7f8c8d" stroke-width="2" marker-end="url(#m)"/>
  <text x="112" y="314" class="rl">WROTE</text>

  <path d="M204,388 L386,300" stroke="#7f8c8d" stroke-width="2" marker-end="url(#m)"/>
  <text x="300" y="356" class="rl">REVIEWS</text>

  <path d="M700,232 L494,232" stroke="#7f8c8d" stroke-width="2" marker-end="url(#m)"/>
  <text x="600" y="222" class="rl">SUPPLIES</text>
  <text x="600" y="246" class="pp">costPrice, leadTimeDays</text>

  <path d="M700,352 L470,282" stroke="#7f8c8d" stroke-width="2" marker-end="url(#m)"/>
  <text x="600" y="332" class="rl">STOCKS</text>
  <text x="600" y="348" class="pp">quantity</text>

  <path d="M700,388 L204,268" stroke="#7f8c8d" stroke-width="2" marker-end="url(#m)" stroke-dasharray="4,3"/>
  <text x="440" y="440" class="rl">SHIPPED_FROM</text>
  <text x="440" y="456" class="pp">carrier, shipDate, onTime</text>

  <!-- nodes -->
  <circle cx="150" cy="84" r="38" fill="#4C9BE8" stroke="#2E6DA4" stroke-width="2"/>
  <text x="150" y="82" class="lb">Customer</text><text x="150" y="96" class="pp" fill="#eaf2fb">1,201</text>

  <circle cx="150" cy="232" r="38" fill="#7FBF7F" stroke="#4F8F4F" stroke-width="2"/>
  <text x="150" y="230" class="lb">Order</text><text x="150" y="244" class="pp" fill="#f0f8f0">6,030</text>

  <circle cx="150" cy="390" r="38" fill="#F2D06B" stroke="#C9A227" stroke-width="2"/>
  <text x="150" y="388" class="lb" fill="#5a4a10">Review</text><text x="150" y="402" class="pp" fill="#6b5a1a">3,570</text>

  <circle cx="440" cy="232" r="38" fill="#F2A65A" stroke="#B87333" stroke-width="2"/>
  <text x="440" y="230" class="lb">Product</text><text x="440" y="244" class="pp" fill="#fff8f0">1,000</text>

  <circle cx="440" cy="84" r="38" fill="#B98FD9" stroke="#8B5FB0" stroke-width="2"/>
  <text x="440" y="82" class="lb">Category</text><text x="440" y="96" class="pp" fill="#f8f0ff">56</text>

  <circle cx="654" cy="84" r="38" fill="#B98FD9" stroke="#8B5FB0" stroke-width="2" opacity="0.45"/>
  <text x="654" y="88" class="lb">(self)</text>

  <circle cx="740" cy="232" r="38" fill="#E86A6A" stroke="#B04040" stroke-width="2"/>
  <text x="740" y="230" class="lb">Supplier</text><text x="740" y="244" class="pp" fill="#fdeaea">150</text>

  <circle cx="740" cy="378" r="38" fill="#5FC9C9" stroke="#3A9797" stroke-width="2"/>
  <text x="740" y="372" class="lb">Warehouse</text><text x="740" y="390" class="pp" fill="#eafafa">5</text>
</svg>

</div>

**7 labels, 8 relationship types, ~12,000 nodes, ~37,000 relationships.**

Notice what *isn't* here: there is no `order_items` box, no `product_suppliers` box, no
`inventory` box. Those three relational join tables **became relationships**. That's three fewer
things to join through — and it's exactly why the graph query in §1.6 was shorter.

## 2.5 Naming conventions

These are conventions, not rules enforced by the database — but every Neo4j codebase you'll
ever join follows them, so start now.

| Element | Convention | Example |
|---|---|---|
| Label | `UpperCamelCase`, **singular** | `:Customer`, `:OrderItem` |
| Relationship type | `UPPER_SNAKE_CASE`, **verb** | `:PLACED`, `:IN_CATEGORY` |
| Property | `lowerCamelCase` | `customerId`, `unitPrice` |
| Variable in a query | short, lowercase | `c`, `p`, `ord` |

**Why singular labels?** Because a pattern should read like a sentence:
`(c:Customer)-[:PLACED]->(o:Order)` reads "a Customer placed an Order." `(:Customers)` would read
wrong.

**Why verbs for relationships?** Same reason — the pattern becomes a readable clause. Prefer
`PLACED` over `ORDER_REL`, and prefer past tense for events (`PLACED`, `WROTE`) with present
tense for states (`SUPPLIES`, `STOCKS`).

## 2.6 Exercises — Part 2

**Exercise 2.1** — Our `customers.csv` has `city`, `state`, `zip_code`, `lat`, `lon`.
Should location be a property of `Customer`, or a separate `(:City)` node? Argue **both** sides,
then state which you'd choose for (a) a marketing dashboard, (b) a fraud investigation platform.

<details>
<summary><b>&#128161; Show solution &mdash; 2.1</b></summary>


**Keep as properties when:** you only filter and group (`WHERE c.state = 'CA'`,
`RETURN c.city, count(*)`). Properties are cheaper, simpler, and aggregation over them is fast.

**Promote to `(:City)` nodes when:** you want to traverse *through* location —
"customers in cities that our top supplier also ships to", or "which cities connect these two
customer clusters". Also useful for a `City -> State -> Country` hierarchy you can traverse
with variable-length paths.

- **(a) Marketing dashboard → properties.** The workload is `GROUP BY city`. A node adds a hop
  and buys nothing.
- **(b) Fraud platform → nodes.** Fraud is about *shared* attributes. If `Address` is a node,
  "five accounts sharing one address" is a one-line pattern:
  `MATCH (a:Address)<-[:LIVES_AT]-(c:Customer) WITH a, count(c) AS n WHERE n > 4 RETURN a, n`.
  As a property, that's an expensive aggregation over every customer.

**The transferable lesson:** *shared identifiers should become nodes when detecting sharing is
the point.*


</details>

**Exercise 2.2** — `payment_transactions.csv` links to `orders`. Should a payment be a node or a
relationship? What about `shipments.csv`?

<details>
<summary><b>&#128161; Show solution &mdash; 2.2</b></summary>


**Payment → node.** A payment has rich state of its own: provider, authorisation code, risk
score, `chargeback_raised`, `is_3d_secure`. It participates in its own workflows (refunds,
disputes, chargebacks — each of which will want to point at the payment). Anything with a
lifecycle wants to be a node.

**Shipment → arguably either.**
- As a **relationship** `(:Order)-[:SHIPPED_FROM {carrier, shipDate, onTime}]->(:Warehouse)`:
  simple, one hop, great if you only ask "which warehouse served this order?"
- As a **node** `(:Order)-[:FULFILLED_BY]->(:Shipment)-[:FROM]->(:Warehouse)`: necessary the
  moment you add tracking scan events, split shipments (one order → several shipments), or
  carrier hand-offs.

We model it as a relationship in this course for simplicity, and Part 11 shows how to *refactor*
it into a node — which is a realistic thing to have to do on a live system.


</details>

**Exercise 2.3** — A relationship connects exactly two nodes. So how would you model
*"Supplier S supplied Product P to Warehouse W on date D"* — a genuinely three-way fact?

<details>
<summary><b>&#128161; Show solution &mdash; 2.3</b></summary>


You **can't** express a 3-way fact as one relationship. Introduce an intermediate
("hyper-edge" or "event") node:

```cypher
(:Supplier)-[:MADE]->(d:Delivery {date: date('2024-03-01'), qty: 500})
(d)-[:OF_PRODUCT]->(:Product)
(d)-[:TO_WAREHOUSE]->(:Warehouse)
```

`Delivery` reifies the event. This pattern shows up constantly:
- Employment: `(Person)-[:HAS]->(Employment)-[:AT]->(Company)`, holding role and dates
- Flights: `(Flight)` linking airline, origin, destination, aircraft
- Prescriptions: linking doctor, patient, drug, date

**The signal to watch for:** whenever a fact involves 3+ entities, or the connection itself has
an identity and lifecycle, you need an intermediate node.


</details>

## 2.7 Checkpoint 2

No code — this is a **conceptual** checkpoint. Tick these honestly before moving on.

- [ ] I can name the four building blocks of a property graph
- [ ] I can explain why direction is stored but doesn't have to be queried
- [ ] I can state the rule for choosing node vs. property
- [ ] I can explain why `order_items` becomes a relationship but `reviews` becomes a node
- [ ] I understand that the "right" model depends on the questions asked

> If any box is unticked, re-read §2.2 and §2.3 before continuing. Part 3 assumes all of this.

---
# Part 3 — Loading Data into Neo4j Aura

## 3.1 Why we don't use `LOAD CSV`

Most Neo4j tutorials teach `LOAD CSV`. We can't use it here, and it's important you know why:

```cypher
LOAD CSV WITH HEADERS FROM 'file:///customers.csv' AS row   -- fails on Aura
```

Aura is a **managed cloud service**. There is no filesystem you can drop a CSV onto, and
`file:///` URLs are disabled. `LOAD CSV` on Aura works only from a **publicly reachable HTTPS
URL** (a public S3 object, a GitHub raw link, and so on).

So we'll use the approach real production systems use anyway: **read the data in Python and
send it over the Bolt driver in batches**. This is the same technique you'd use to ingest from
Kafka, a REST API, Databricks, or an existing relational database — so it's the more
transferable skill.

## 3.2 Order of operations (this order matters)

1. **Clear** any previous data — makes the notebook re-runnable
2. **Create constraints** — *before* loading, never after
3. **Load nodes** — all of them, before any relationships
4. **Load relationships** — now that both endpoints are guaranteed to exist
5. **Verify** counts

### Why constraints before data?

A uniqueness constraint in Neo4j **automatically creates a backing index**. Our relationship
loader runs `MATCH (c:Customer {customerId: row.cid})` — 6,030 times for orders alone.

- **With** the constraint: each lookup is an index seek, microseconds.
- **Without** it: each lookup is a **full label scan** of all 1,201 customers.

That's the difference between a load that takes 30 seconds and one that takes 20 minutes.
Creating the index *afterwards* doesn't help — the damage is done during the load.

> **This is the single most common cause of "Neo4j is slow" complaints from beginners.**

In [ ]:
import pandas as pd
import numpy as np
import pathlib

DATA = pathlib.Path("ecommerce_graph_data")


def prep(df, dates=(), dateonly=()):
    """Convert a DataFrame into a list of dicts the Neo4j driver accepts.

    Three jobs:
      1. NaN -> None   (Neo4j has no NaN; None becomes a missing property)
      2. numpy types -> native Python (the driver can't serialise np.int64)
      3. '2024-10-27 10:42:00' -> '2024-10-27T10:42:00'  (ISO-8601 for datetime())
    """
    df = df.copy()
    for col in dates:
        if col in df:
            df[col] = (df[col].astype("object").where(df[col].notna(), None)
                       .map(lambda v: str(v).replace(" ", "T") if v is not None else None))
    for col in dateonly:
        if col in df:
            df[col] = (df[col].astype("object").where(df[col].notna(), None)
                       .map(lambda v: str(v)[:10] if v is not None else None))
    df = df.replace({np.nan: None})
    records = df.to_dict("records")
    for rec in records:
        for k, v in rec.items():
            if isinstance(v, np.integer):
                rec[k] = int(v)
            elif isinstance(v, np.floating):
                rec[k] = None if pd.isna(v) else float(v)
            elif isinstance(v, np.bool_):
                rec[k] = bool(v)
            elif pd.isna(v) if not isinstance(v, (list, dict)) else False:
                rec[k] = None
    return records


def load(cypher, rows, batch_size=1000, label=""):
    """Send rows to Neo4j in batches using UNWIND.

    Why batch at all? Two reasons:
      * One transaction per row = one network round-trip each. Catastrophically slow.
      * One transaction for ALL rows = the whole change set must fit in memory,
        and Aura Free has a 1 GB heap. It will fall over.
    A batch of ~1,000 is the usual sweet spot.
    """
    import time
    total, t0 = 0, time.perf_counter()
    for i in range(0, len(rows), batch_size):
        chunk = rows[i:i + batch_size]
        gdb.run(cypher, {"rows": chunk})
        total += len(chunk)
        print("\r   {:<24s} {:>6,}/{:,}".format(label, total, len(rows)), end="")
    print("\r   {:<24s} {:>6,}/{:,}   {:5.1f}s".format(
        label, total, len(rows), time.perf_counter() - t0))


print("Loader helpers defined.")

### The `UNWIND` idiom — the most important loading pattern in Cypher

```cypher
UNWIND $rows AS row
MERGE (c:Customer {customerId: row.customer_id})
SET   c.firstName = row.first_name
```

`UNWIND` takes a **list** and turns it into **rows**. It is the exact inverse of `collect()`.

So we send 1,000 customers as *one parameter* in *one network round-trip*, and Cypher expands
them into 1,000 rows internally. Compare:

| Approach | Round-trips for 6,030 orders | Plan compilations |
|---|---|---|
| One query per row | 6,030 | 6,030 (or cached) |
| `UNWIND` batches of 1,000 | **7** | **1** |

That's not a small optimisation — it's usually **100x faster**.

In [ ]:
# ------------------------------------------------------------------ STEP 1: clear
# DETACH DELETE removes a node AND all its relationships. Plain DELETE
# refuses to delete a node that still has relationships - a good safety
# feature that stops you silently orphaning edges.
#
# We delete in batches because removing 37k relationships in a single
# transaction would exceed the Aura Free heap.

print("Clearing existing data...")
while True:
    deleted = gdb.run("""
        MATCH (n)
        WITH n LIMIT 5000
        DETACH DELETE n
        RETURN count(n) AS deleted
    """).iloc[0]["deleted"]
    if deleted == 0:
        break
    print("   deleted {:,} nodes".format(deleted))

remaining = gdb.run("MATCH (n) RETURN count(n) AS n").iloc[0]["n"]
print("Database now holds {} nodes.".format(remaining))

In [ ]:
# ------------------------------------------------- STEP 2: constraints & indexes
# Syntax note (Neo4j 5+):
#   CREATE CONSTRAINT <name> IF NOT EXISTS FOR (n:Label) REQUIRE n.prop IS UNIQUE
# The older Neo4j 3/4 syntax (ON ... ASSERT) has been removed. If you find it in
# a blog post, that blog post is out of date.

constraints = [
    "CREATE CONSTRAINT customer_id  IF NOT EXISTS FOR (n:Customer)  REQUIRE n.customerId  IS UNIQUE",
    "CREATE CONSTRAINT product_id   IF NOT EXISTS FOR (n:Product)   REQUIRE n.productId   IS UNIQUE",
    "CREATE CONSTRAINT order_id     IF NOT EXISTS FOR (n:Order)     REQUIRE n.orderId     IS UNIQUE",
    "CREATE CONSTRAINT category_id  IF NOT EXISTS FOR (n:Category)  REQUIRE n.categoryId  IS UNIQUE",
    "CREATE CONSTRAINT supplier_id  IF NOT EXISTS FOR (n:Supplier)  REQUIRE n.supplierId  IS UNIQUE",
    "CREATE CONSTRAINT warehouse_id IF NOT EXISTS FOR (n:Warehouse) REQUIRE n.warehouseId IS UNIQUE",
    "CREATE CONSTRAINT review_id    IF NOT EXISTS FOR (n:Review)    REQUIRE n.reviewId    IS UNIQUE",
]

# Plain indexes for properties we filter on often but which are not unique.
indexes = [
    "CREATE INDEX customer_segment IF NOT EXISTS FOR (n:Customer) ON (n.segment)",
    "CREATE INDEX customer_city    IF NOT EXISTS FOR (n:Customer) ON (n.city)",
    "CREATE INDEX product_brand    IF NOT EXISTS FOR (n:Product)  ON (n.brand)",
    "CREATE INDEX product_price    IF NOT EXISTS FOR (n:Product)  ON (n.price)",
    "CREATE INDEX order_status     IF NOT EXISTS FOR (n:Order)    ON (n.status)",
    "CREATE INDEX order_date       IF NOT EXISTS FOR (n:Order)    ON (n.orderDate)",
    "CREATE INDEX category_name    IF NOT EXISTS FOR (n:Category) ON (n.name)",
]

for stmt in constraints + indexes:
    gdb.run(stmt)

print("{} constraints + {} indexes created.\n".format(len(constraints), len(indexes)))
gdb.run("SHOW CONSTRAINTS YIELD name, labelsOrTypes, properties, type RETURN *")

### What did those constraints actually do?

An `IS UNIQUE` constraint does **two** things:

1. **Enforces** that no two `:Customer` nodes share a `customerId` — a violating write fails.
2. **Creates a range index** on `(:Customer).customerId` as a side effect.

The second point is why we create constraints *first*. Point 1 is a correctness guarantee;
point 2 is the performance guarantee.

> **Neo4j has no auto-increment primary key.** Every node has an internal ID, but it is *not
> stable* — Neo4j may reuse the ID of a deleted node. **Never store or reference internal IDs
> in your application.** Always define your own business key with a uniqueness constraint,
> exactly as we've done here.

In [ ]:
# ------------------------------------------------------------------ STEP 3: nodes
print("Loading nodes\n" + "-" * 62)

# ---- Customer
customers = prep(pd.read_csv(DATA / "customers.csv"),
                 dates=["created_at", "last_order_date"], dateonly=["date_of_birth"])
load("""
UNWIND $rows AS row
MERGE (c:Customer {customerId: row.customer_id})
SET c.firstName       = row.first_name,
    c.lastName        = row.last_name,
    c.name            = row.first_name + ' ' + row.last_name,
    c.email           = row.email,
    c.city            = row.city,
    c.state           = row.state,
    c.country         = row.country,
    c.segment         = row.customer_segment,
    c.channel         = row.acquisition_channel,
    c.loyaltyPoints   = toInteger(row.loyalty_points),
    c.lifetimeOrders  = toInteger(row.lifetime_orders),
    c.lifetimeRevenue = toFloat(row.lifetime_revenue),
    c.preferredDevice = row.preferred_device,
    c.newsletter      = row.is_newsletter_subscribed,
    c.createdAt       = datetime(row.created_at),
    c.dateOfBirth     = date(row.date_of_birth),
    c.location        = point({latitude: toFloat(row.lat), longitude: toFloat(row.lon)})
""", customers, label="Customer")

# ---- Category
categories = prep(pd.read_csv(DATA / "categories.csv"), dates=["created_at"])
load("""
UNWIND $rows AS row
MERGE (c:Category {categoryId: toInteger(row.category_id)})
SET c.name        = row.category_name,
    c.level       = toInteger(row.category_level),
    c.description = row.description
""", categories, label="Category")

# ---- Product
products = prep(pd.read_csv(DATA / "products.csv"), dates=["created_at"])
load("""
UNWIND $rows AS row
MERGE (p:Product {productId: row.product_id})
SET p.name          = row.product_name,
    p.brand         = row.brand,
    p.sku           = row.sku,
    p.price         = toFloat(row.price),
    p.costPrice     = toFloat(row.cost_price),
    p.marginPct     = toFloat(row.margin_pct),
    p.weightKg      = toFloat(row.weight_kg),
    p.avgRating     = toFloat(row.avg_rating),
    p.reviewCount   = toInteger(row.review_count),
    p.returnRate    = toFloat(row.return_rate_pct),
    p.isFeatured    = row.is_featured,
    p.isActive      = row.is_active,
    p.stockQuantity = toInteger(row.stock_quantity),
    p.tags          = [t IN split(coalesce(row.tags, ''), ',') | trim(t)],
    p.createdAt     = datetime(row.created_at)
""", products, label="Product")

# ---- Order
orders = prep(pd.read_csv(DATA / "orders.csv"), dates=["order_date"])
load("""
UNWIND $rows AS row
MERGE (o:Order {orderId: row.order_id})
SET o.orderDate      = datetime(row.order_date),
    o.totalAmount    = toFloat(row.total_amount),
    o.status         = row.order_status,
    o.paymentMethod  = row.payment_method,
    o.shippingCity   = row.shipping_city,
    o.shippingState  = row.shipping_state,
    o.deviceType     = row.device_type,
    o.shippingMethod = row.shipping_method,
    o.shippingCost   = toFloat(row.shipping_cost),
    o.discountAmount = toFloat(row.discount_amount),
    o.couponCode     = row.coupon_code,
    o.referralSource = row.referral_source,
    o.isGift         = row.is_gift
""", orders, label="Order")

# ---- Supplier
suppliers = prep(pd.read_csv(DATA / "suppliers.csv"), dates=["created_at"])
load("""
UNWIND $rows AS row
MERGE (s:Supplier {supplierId: row.supplier_id})
SET s.name         = row.supplier_name,
    s.contact      = row.contact_person,
    s.email        = row.email,
    s.city         = row.city,
    s.state        = row.state,
    s.country      = row.country,
    s.paymentTerms = row.payment_terms,
    s.rating       = toFloat(row.rating)
""", suppliers, label="Supplier")

# ---- Warehouse
warehouses = prep(pd.read_csv(DATA / "warehouses.csv"), dateonly=["operating_since"])
load("""
UNWIND $rows AS row
MERGE (w:Warehouse {warehouseId: toInteger(row.warehouse_id)})
SET w.name         = row.warehouse_name,
    w.city         = row.city,
    w.state        = row.state,
    w.capacitySqft = toInteger(row.capacity_sqft),
    w.manager      = row.manager_name,
    w.location     = point({latitude: toFloat(row.lat), longitude: toFloat(row.lon)})
""", warehouses, label="Warehouse")

# ---- Review
reviews = prep(pd.read_csv(DATA / "reviews.csv"), dates=["review_date"])
load("""
UNWIND $rows AS row
MERGE (r:Review {reviewId: row.review_id})
SET r.rating       = toInteger(row.rating),
    r.title        = row.review_title,
    r.text         = row.review_text,
    r.sentiment    = row.sentiment,
    r.helpfulVotes = toInteger(row.helpful_votes),
    r.verified     = row.verified_purchase = 'Yes',
    r.reviewDate   = datetime(row.review_date)
""", reviews, label="Review")

print("-" * 62)
gdb.run("MATCH (n) UNWIND labels(n) AS label "
        "RETURN label, count(*) AS nodes ORDER BY nodes DESC")

**Expected output:** 7 labels — Order 6,030 · Review 3,570 · Customer 1,201 · Product 1,000 ·
Supplier 150 · Category 56 · Warehouse 5. **Total 12,012 nodes.**

### Three Cypher techniques worth pausing on

**`MERGE` vs `CREATE`** — `CREATE` always makes a new node. `MERGE` is "match, or create if
absent" — an upsert. We use `MERGE` so this notebook is safely re-runnable: run the load twice
and you still get 1,201 customers, not 2,402.

> **`MERGE` matches on the *entire* pattern you give it.**
> `MERGE (c:Customer {customerId: 1, name: 'Bob'})` looks for a customer with **both** properties.
> If one exists with `customerId: 1` but a different name, MERGE creates a **second** node —
> and only then does your uniqueness constraint save you by throwing an error.
> **Always `MERGE` on the business key alone, then `SET` everything else.** That's the pattern above.

**`point({latitude: .., longitude: ..})`** — Neo4j has a native spatial type. Now
`point.distance(a, b)` returns real metres, and a point index makes proximity search fast.

**`[t IN split(row.tags, ',') | trim(t)]`** — a *list comprehension*. It splits the string
`"new-arrival, cushioned"` into a list and trims each element, storing a genuine list property.
`coalesce(row.tags, '')` guards against nulls. More on comprehensions in Part 7.

In [ ]:
# --------------------------------------------------------- STEP 4: relationships
print("Loading relationships\n" + "-" * 62)

# ---- Category hierarchy (a self-relationship). Only 46 of 56 have a parent.
cat_parents = [r for r in prep(pd.read_csv(DATA / "categories.csv"))
               if r["parent_category_id"] is not None]
load("""
UNWIND $rows AS row
MATCH (child:Category  {categoryId: toInteger(row.category_id)})
MATCH (parent:Category {categoryId: toInteger(row.parent_category_id)})
MERGE (parent)-[:PARENT_OF]->(child)
""", cat_parents, label="PARENT_OF")

# ---- Product -> Category.  The CSV joins on category NAME, not id.
prod_cat = prep(pd.read_csv(DATA / "products.csv")[["product_id", "category"]])
load("""
UNWIND $rows AS row
MATCH (p:Product  {productId: row.product_id})
MATCH (c:Category {name: row.category})
MERGE (p)-[:IN_CATEGORY]->(c)
""", prod_cat, label="IN_CATEGORY")

# ---- Customer -> Order
cust_ord = prep(pd.read_csv(DATA / "orders.csv")[["order_id", "customer_id"]])
load("""
UNWIND $rows AS row
MATCH (c:Customer {customerId: row.customer_id})
MATCH (o:Order    {orderId:    row.order_id})
MERGE (c)-[:PLACED]->(o)
""", cust_ord, label="PLACED")

# ---- Order -> Product, with properties ON the relationship
order_items = prep(pd.read_csv(DATA / "order_items.csv"))
load("""
UNWIND $rows AS row
MATCH (o:Order   {orderId:   row.order_id})
MATCH (p:Product {productId: row.product_id})
MERGE (o)-[ci:CONTAINS]->(p)
SET ci.quantity   = toInteger(row.quantity),
    ci.unitPrice  = toFloat(row.unit_price),
    ci.totalPrice = toFloat(row.total_price)
""", order_items, label="CONTAINS")

# ---- Customer -> Review -> Product
rev_links = prep(pd.read_csv(DATA / "reviews.csv")[["review_id", "customer_id", "product_id"]])
load("""
UNWIND $rows AS row
MATCH (r:Review   {reviewId:   row.review_id})
MATCH (c:Customer {customerId: row.customer_id})
MATCH (p:Product  {productId:  row.product_id})
MERGE (c)-[:WROTE]->(r)
MERGE (r)-[:REVIEWS]->(p)
""", rev_links, label="WROTE / REVIEWS")

# ---- Supplier -> Product
prod_sup = prep(pd.read_csv(DATA / "product_suppliers.csv"))
load("""
UNWIND $rows AS row
MATCH (s:Supplier {supplierId: row.supplier_id})
MATCH (p:Product  {productId:  row.product_id})
MERGE (s)-[sup:SUPPLIES]->(p)
SET sup.costPrice    = toFloat(row.cost_price),
    sup.leadTimeDays = toInteger(row.lead_time_days),
    sup.minOrderQty  = toInteger(row.minimum_order_quantity),
    sup.isPrimary    = row.is_primary_supplier = 'Yes'
""", prod_sup, label="SUPPLIES")

# ---- Warehouse -> Product
inventory = prep(pd.read_csv(DATA / "inventory.csv"))
load("""
UNWIND $rows AS row
MATCH (w:Warehouse {warehouseId: toInteger(row.warehouse_id)})
MATCH (p:Product   {productId:   row.product_id})
MERGE (w)-[st:STOCKS]->(p)
SET st.quantityAvailable = toInteger(row.quantity_available),
    st.quantityReserved  = toInteger(row.quantity_reserved),
    st.reorderPoint      = toInteger(row.reorder_point),
    st.binLocation       = row.bin_location
""", inventory, label="STOCKS")

# ---- Order -> Warehouse
shipments = prep(pd.read_csv(DATA / "shipments.csv"),
                 dates=["ship_date", "actual_delivery"])
load("""
UNWIND $rows AS row
MATCH (o:Order     {orderId:     row.order_id})
MATCH (w:Warehouse {warehouseId: toInteger(row.warehouse_id)})
MERGE (o)-[sh:SHIPPED_FROM]->(w)
SET sh.carrier      = row.carrier,
    sh.shipDate     = datetime(row.ship_date),
    sh.deliveryDays = toInteger(row.delivery_days_actual),
    sh.onTime       = row.on_time,
    sh.shippingCost = toFloat(row.shipping_cost)
""", shipments, label="SHIPPED_FROM")

print("-" * 62)
gdb.run("MATCH ()-[r]->() RETURN type(r) AS relationship, count(*) AS count "
        "ORDER BY count DESC")

**Expected output:** CONTAINS 14,456 · PLACED 6,030 · SHIPPED_FROM 4,810 · REVIEWS 3,570 ·
WROTE 3,570 · SUPPLIES 1,679 · STOCKS 1,643 · IN_CATEGORY 1,000 · PARENT_OF 46.
**Total ≈ 36,804 relationships.**

## 3.3 See your graph

Every load script should end by *looking* at the data. Let's pull one customer's neighbourhood.

In [ ]:
# Find the customer with the most orders, then draw their world.
top = gdb.run("""
    MATCH (c:Customer)-[:PLACED]->(o:Order)
    RETURN c.customerId AS cid, c.name AS name, count(o) AS orders
    ORDER BY orders DESC LIMIT 1
""")
CID = int(top.iloc[0]["cid"])
print("Busiest customer: {} (id {}) with {} orders\n".format(
    top.iloc[0]["name"], CID, top.iloc[0]["orders"]))

draw("""
    MATCH path = (c:Customer {customerId: $cid})-[:PLACED]->(o:Order)-[:CONTAINS]->(p:Product)
    WITH path LIMIT 25
    RETURN path
""", {"cid": CID}, title="Customer {}: orders and the products inside them".format(CID))

In [ ]:
# The category hierarchy is a genuine tree - let's see it.
draw("""
    MATCH path = (root:Category {level: 1})-[:PARENT_OF*1..2]->(:Category)
    WHERE root.name IN ['Electronics', 'Clothing & Apparel']
    RETURN path
""", title="Category hierarchy: two roots, three levels", figsize=(14, 9))

## 3.4 Verify the load is *correct*, not merely complete

Counting rows proves you loaded *something*. It doesn't prove you loaded it *correctly*.
The real test: **can I reconstruct a known fact by traversing the graph?**

In [ ]:
# Cross-check: do the line items reconstruct sensible order subtotals?
check = gdb.run("""
    MATCH (o:Order)-[c:CONTAINS]->(:Product)
    WITH o, round(sum(c.totalPrice), 2) AS lineSum, count(*) AS lines
    RETURN count(*)             AS ordersChecked,
           round(avg(lines), 2) AS avgLinesPerOrder,
           min(lineSum)         AS minLineSum,
           max(lineSum)         AS maxLineSum
""")
display(check)

# Every order must belong to exactly one customer - orphans mean a broken load.
display(gdb.run("""
    MATCH (o:Order) WHERE NOT (:Customer)-[:PLACED]->(o)
    RETURN count(o) AS orphanOrders
"""))

# Every product must sit in a category.
display(gdb.run("""
    MATCH (p:Product) WHERE NOT (p)-[:IN_CATEGORY]->(:Category)
    RETURN count(p) AS uncategorisedProducts
"""))

## 3.5 Checkpoint 3

In [ ]:
n = lambda q: gdb.run(q).iloc[0, 0]

checkpoint("Part 3 - Data loaded", [
    ("1,201 Customer nodes",      n("MATCH (n:Customer)  RETURN count(n)") == 1201),
    ("1,000 Product nodes",       n("MATCH (n:Product)   RETURN count(n)") == 1000),
    ("6,030 Order nodes",         n("MATCH (n:Order)     RETURN count(n)") == 6030),
    ("56 Category nodes",         n("MATCH (n:Category)  RETURN count(n)") == 56),
    ("3,570 Review nodes",        n("MATCH (n:Review)    RETURN count(n)") == 3570),
    ("14,456 CONTAINS rels",      n("MATCH ()-[r:CONTAINS]->() RETURN count(r)") == 14456),
    ("No orphan orders",
     n("MATCH (o:Order) WHERE NOT (:Customer)-[:PLACED]->(o) RETURN count(o)") == 0),
    ("No uncategorised products",
     n("MATCH (p:Product) WHERE NOT (p)-[:IN_CATEGORY]->() RETURN count(p)") == 0),
    ("7 uniqueness constraints",  len(gdb.run("SHOW CONSTRAINTS")) >= 7),
])

---
# Part 4 — Cypher Fundamentals

## 4.1 The mental model

Cypher is a **pattern-matching** language. You don't tell the database *how* to find data
(as you would by writing joins) — you **draw the shape you want**, and the database finds every
subgraph matching that shape.

```
      (  )-[  ]->(  )
       |     |     |
      node  rel   node
```

Every Cypher query is built from a handful of clauses:

| Clause | Job | SQL analogue |
|---|---|---|
| `MATCH` | Find a pattern | `FROM` + `JOIN` |
| `WHERE` | Filter the matches | `WHERE` |
| `WITH` | Pipe results into the next stage | `GROUP BY` / subquery |
| `RETURN` | Choose what comes back | `SELECT` |
| `ORDER BY` / `SKIP` / `LIMIT` | Sort and page | same |

## 4.2 Your first queries

In [ ]:
# The simplest possible query: find 5 customers.
# (c:Customer) means "a node, call it c, that has the label Customer"
gdb.run("""
    MATCH (c:Customer)
    RETURN c.name AS name, c.city AS city, c.segment AS segment
    LIMIT 5
""", show_time=True)

> **Always use `LIMIT` while exploring.** `MATCH (n) RETURN n` on a large graph will try to
> return everything. `LIMIT` is your seatbelt.

In [ ]:
# Filter with WHERE.
gdb.run("""
    MATCH (p:Product)
    WHERE p.price > 900 AND p.avgRating >= 4.5
    RETURN p.name AS product, p.brand AS brand,
           p.price AS price, p.avgRating AS rating
    ORDER BY p.price DESC
    LIMIT 10
""", show_time=True)

In [ ]:
# Inline property matching is shorthand for equality only.
# These two queries are IDENTICAL:
a = gdb.run("MATCH (c:Customer {segment:'VIP', preferredDevice:'mobile'}) RETURN count(c) AS n")
b = gdb.run("MATCH (c:Customer) WHERE c.segment='VIP' AND c.preferredDevice='mobile' "
            "RETURN count(c) AS n")
print("inline form :", int(a.iloc[0]["n"]))
print("WHERE form  :", int(b.iloc[0]["n"]))
print("\nUse the inline form for equality; use WHERE for anything else (>, <, IN, STARTS WITH).")

## 4.3 Traversing your first relationship

This is where Cypher starts to earn its keep.

In [ ]:
# Which products did a specific customer buy?
# Read the pattern out loud:
#   "a Customer PLACED an Order which CONTAINS a Product"
gdb.run("""
    MATCH (c:Customer {customerId: $cid})-[:PLACED]->(o:Order)-[:CONTAINS]->(p:Product)
    RETURN o.orderId AS orderId, o.orderDate AS date,
           p.name AS product, p.price AS price
    ORDER BY o.orderDate DESC
    LIMIT 10
""", {"cid": CID}, show_time=True)

### Anatomy of that pattern

```
MATCH (c:Customer {customerId: $cid})-[:PLACED]->(o:Order)-[:CONTAINS]->(p:Product)
      |________ anchor ____________|  |_ rel _|  |_node_|  |__ rel __|  |_ node _|
```

- **`(c:Customer {customerId: $cid})`** is the **anchor**. Because `customerId` has a unique
  constraint, Neo4j starts with a single index seek — one node — and expands outward from there.
  Choosing a good anchor is the number one performance lever in Cypher.
- **`-[:PLACED]->`** follows relationships of type `PLACED`, in the direction of the arrow.
- **`$cid`** is a **parameter**. Never build queries by string concatenation: parameters let
  Neo4j reuse the compiled plan and eliminate injection risk.

### Direction: the three forms

| Pattern | Meaning |
|---|---|
| `(a)-[:R]->(b)` | Follow `R` **outgoing** from `a` |
| `(a)<-[:R]-(b)` | Follow `R` **incoming** to `a` |
| `(a)-[:R]-(b)` | **Either direction** — and there is no performance penalty |

Because our model stores `(Customer)-[:PLACED]->(Order)`, finding the customer *from* an order
uses the reversed arrow: `(o:Order)<-[:PLACED]-(c:Customer)`.

In [ ]:
# Reverse traversal: who bought the most expensive products?
gdb.run("""
    MATCH (p:Product)<-[:CONTAINS]-(:Order)<-[:PLACED]-(c:Customer)
    WHERE p.price > 1400
    RETURN p.name AS product, p.price AS price,
           c.name AS customer, c.segment AS segment
    ORDER BY p.price DESC
    LIMIT 10
""", show_time=True)

### Anonymous nodes and relationships

Notice `(:Order)` in that query — no variable name. If you never reference a node again,
**don't name it**. This isn't just cosmetic: naming a variable forces Neo4j to carry it in the
result row, which costs memory.

You can go further with a completely empty node: `()`.

```cypher
MATCH (p:Product)<-[:CONTAINS]-()<-[:PLACED]-(c:Customer)   // also valid
```

Prefer `(:Order)` over `()` though — the label documents your intent *and* helps the planner.

## 4.4 The `WHERE` toolbox

Cypher's `WHERE` is richer than SQL's in one important way: **it can filter on patterns**.

In [ ]:
# String matching
gdb.run("""
    MATCH (p:Product)
    WHERE p.name STARTS WITH 'Smart'
       OR p.name ENDS WITH 'Pro'
       OR p.name CONTAINS 'Wireless'
    RETURN p.name AS product, p.brand AS brand, p.price AS price
    ORDER BY p.price DESC LIMIT 8
""")

In [ ]:
# Ranges, IN lists, regex and NULL handling all at once
gdb.run("""
    MATCH (c:Customer)
    WHERE c.loyaltyPoints >= 5000 AND c.loyaltyPoints <= 15000
      AND c.segment IN ['VIP', 'Regular']
      AND c.email =~ '.*@example\\.com'
      AND c.city IS NOT NULL
    RETURN c.name AS name, c.segment AS segment,
           c.loyaltyPoints AS points, c.city AS city
    ORDER BY c.loyaltyPoints DESC LIMIT 8
""")

### `WHERE` on patterns — the superpower

You can put a **pattern** inside `WHERE` to mean "such that this connection exists (or doesn't)".
SQL needs `EXISTS (SELECT ...)` or a `LEFT JOIN ... IS NULL` for the same thing.

In [ ]:
# Products that have NEVER been ordered.
# The pattern in WHERE acts as an existence test - no join required.
never_ordered = gdb.run("""
    MATCH (p:Product)
    WHERE NOT (p)<-[:CONTAINS]-(:Order)
    RETURN p.name AS product, p.brand AS brand, p.price AS price
    ORDER BY p.price DESC
""")
print("{} products have never been ordered\n".format(len(never_ordered)))
never_ordered.head(10)

In [ ]:
# Customers who wrote a review but placed no orders at all
gdb.run("""
    MATCH (c:Customer)
    WHERE (c)-[:WROTE]->(:Review)
      AND NOT (c)-[:PLACED]->(:Order)
    RETURN c.name AS customer, c.segment AS segment
    LIMIT 10
""")

### `NULL` semantics — read this carefully

Neo4j is **schema-optional**: a property that was never set simply *doesn't exist* on that node.
Reading it returns `null`, and `null` behaves as it does in SQL:

| Expression | Result | Why |
|---|---|---|
| `null = null` | `null` | Not `true`! Unknown equals unknown is unknown |
| `null <> null` | `null` | Same reason |
| `n.missing IS NULL` | `true` | The **correct** way to test |
| `null + 5` | `null` | Nulls propagate through arithmetic |
| `WHERE n.missing > 5` | row is **dropped** | `null` is not `true`, so the filter fails |
| `coalesce(n.missing, 0)` | `0` | Your null-handling workhorse |

> **The trap:** `WHERE o.discountAmount < 10` silently drops every order where the property was
> never set. If you meant "including orders with no discount", write
> `WHERE coalesce(o.discountAmount, 0) < 10`.

In [ ]:
# Demonstrate the null trap concretely.
# Many of our orders have no couponCode at all.
gdb.run("""
    MATCH (o:Order)
    RETURN count(*)              AS allOrders,
           count(o.couponCode)   AS withCoupon,
           count(*) - count(o.couponCode) AS withoutCoupon
""")

> **`count(o.couponCode)` counts only non-null values**, whereas `count(*)` counts rows.
> This is identical to SQL, and it's a genuinely useful idiom for "how many nodes have this
> property set?"

## 4.5 `RETURN`, `ORDER BY`, `SKIP`, `LIMIT`, `DISTINCT`

In [ ]:
# DISTINCT removes duplicate ROWS (not duplicate values within one column).
# Without it, a customer who bought five Nike items appears five times.
gdb.run("""
    MATCH (c:Customer)-[:PLACED]->(:Order)-[:CONTAINS]->(p:Product)
    WHERE p.brand = 'Nike'
    RETURN DISTINCT c.name AS customer, c.city AS city
    ORDER BY customer
    LIMIT 10
""")

In [ ]:
# Pagination with SKIP + LIMIT - page 3, five rows per page.
PAGE, SIZE = 3, 5
gdb.run("""
    MATCH (p:Product)
    RETURN p.name AS product, p.price AS price
    ORDER BY p.price DESC, p.name      // tie-break: without it, paging is unstable
    SKIP $skip LIMIT $limit
""", {"skip": (PAGE - 1) * SIZE, "limit": SIZE})

> **Always add a tie-breaker to `ORDER BY` when paginating.** If two products share a price and
> the sort is ambiguous, the same row can appear on page 2 *and* page 3 while another is skipped
> entirely. Sorting by `price DESC, name` makes the ordering total and deterministic.

In [ ]:
# Return whole nodes, computed expressions and relationship properties together.
gdb.run("""
    MATCH (c:Customer)-[:PLACED]->(o:Order)-[ci:CONTAINS]->(p:Product)
    WHERE o.status = 'Delivered'
    RETURN c.name                     AS customer,
           p.name                     AS product,
           ci.quantity                AS qty,
           ci.unitPrice               AS unitPrice,
           ci.quantity * ci.unitPrice AS lineTotal,
           p.price - p.costPrice      AS unitMargin,
           labels(c)                  AS customerLabels,
           o.orderDate.year           AS year
    ORDER BY lineTotal DESC
    LIMIT 8
""")

Notice `o.orderDate.year` — temporal values are **structured types**, so you reach into their
components directly: `.year`, `.month`, `.day`, `.hour`, `.quarter`, `.weekday`, `.dayOfWeek`.
No `EXTRACT()` function needed.

## 4.6 Exercises — Part 4

Write and run each query yourself before opening the solution.

**Exercise 4.1** — Find the 10 cheapest **active** products (`isActive = true`) whose average
rating is at least 4.0. Return name, brand, price and rating.

<details>
<summary><b>&#128161; Show solution &mdash; 4.1</b></summary>

```cypher
MATCH (p:Product)
WHERE p.isActive = true AND p.avgRating >= 4.0
RETURN p.name AS product, p.brand AS brand,
       p.price AS price, p.avgRating AS rating
ORDER BY p.price ASC
LIMIT 10
```
**Note:** `WHERE p.isActive` alone also works, since the property is already boolean.
Writing `= true` is more explicit, and safer if the property might be missing on some nodes.

</details>

**Exercise 4.2** — Find all `VIP` customers who have **never written a review**. Return name and
lifetime revenue, highest first, limit 10.

<details>
<summary><b>&#128161; Show solution &mdash; 4.2</b></summary>

```cypher
MATCH (c:Customer {segment: 'VIP'})
WHERE NOT (c)-[:WROTE]->(:Review)
RETURN c.name AS customer, c.lifetimeRevenue AS revenue
ORDER BY revenue DESC
LIMIT 10
```
This is the **anti-join** pattern. In SQL it would be
`LEFT JOIN reviews ... WHERE reviews.id IS NULL` or `NOT EXISTS (SELECT ...)`. In Cypher it's a
negated pattern — far easier to read, and the planner turns it into an efficient anti-semi-apply.

</details>

**Exercise 4.3** — Find every product supplied by a supplier based in state `'CA'`. Return the
product name, supplier name, and the `costPrice` stored **on the relationship**.

<details>
<summary><b>&#128161; Show solution &mdash; 4.3</b></summary>

```cypher
MATCH (s:Supplier {state: 'CA'})-[sup:SUPPLIES]->(p:Product)
RETURN p.name AS product, s.name AS supplier,
       sup.costPrice AS costPrice, sup.leadTimeDays AS leadDays
ORDER BY costPrice DESC
LIMIT 15
```
The key skill is **naming the relationship** (`sup`) so you can read its properties.
Beginners often forget that relationships carry data at all.

</details>

**Exercise 4.4** — Find orders placed in 2024 with status `'Returned'` that shipped from the
warehouse in `Newark`. Return order id, date, amount and the carrier (a relationship property).

<details>
<summary><b>&#128161; Show solution &mdash; 4.4</b></summary>

```cypher
MATCH (o:Order)-[sh:SHIPPED_FROM]->(w:Warehouse {city: 'Newark'})
WHERE o.status = 'Returned' AND o.orderDate.year = 2024
RETURN o.orderId AS orderId, o.orderDate AS date,
       o.totalAmount AS amount, sh.carrier AS carrier, sh.onTime AS onTime
ORDER BY date DESC
LIMIT 15
```
Two things to note: the year filter uses the temporal accessor `.year` rather than string
comparison, and `carrier`/`onTime` live on the **relationship**, not on either node.

</details>

**Exercise 4.5** *(harder)* — Find products stocked in a warehouse where `quantityAvailable` has
fallen **below** the `reorderPoint`. Return product, warehouse, available quantity, reorder point
and the shortfall.

<details>
<summary><b>&#128161; Show solution &mdash; 4.5</b></summary>

```cypher
MATCH (w:Warehouse)-[st:STOCKS]->(p:Product)
WHERE st.quantityAvailable < st.reorderPoint
RETURN p.name                                 AS product,
       w.name                                 AS warehouse,
       st.quantityAvailable                   AS available,
       st.reorderPoint                        AS reorderPoint,
       st.reorderPoint - st.quantityAvailable AS shortfall
ORDER BY shortfall DESC
LIMIT 15
```
Both sides of the comparison are *relationship* properties — a useful reminder that `WHERE`
operates on relationships exactly as it does on nodes.

</details>

## 4.7 Checkpoint 4

In [ ]:
try:
    q1 = gdb.run("MATCH (p:Product) WHERE p.isActive AND p.avgRating >= 4.0 "
                 "RETURN p ORDER BY p.price LIMIT 10")
    q2 = gdb.run("MATCH (c:Customer {segment:'VIP'}) WHERE NOT (c)-[:WROTE]->(:Review) "
                 "RETURN c LIMIT 10")
    q3 = gdb.run("MATCH (:Supplier {state:'CA'})-[s:SUPPLIES]->(p:Product) "
                 "RETURN p.name AS n, s.costPrice AS c LIMIT 5")
    q4 = gdb.run("MATCH (w:Warehouse)-[st:STOCKS]->(p:Product) "
                 "WHERE st.quantityAvailable < st.reorderPoint RETURN count(*) AS n")
    ran = True
except Exception as exc:
    print(exc)
    ran = False

checkpoint("Part 4 - Cypher fundamentals", [
    ("Filtered product query runs",      ran and len(q1) > 0),
    ("Negated-pattern query runs",       ran and len(q2) >= 0),
    ("Relationship properties readable", ran and len(q3) > 0),
    ("Below-reorder-point items found",  ran and int(q4.iloc[0]["n"]) > 0),
])

---
# Part 5 — Aggregation and the `WITH` Pipeline

## 5.1 Implicit grouping — the one thing that confuses everybody

In SQL you must write `GROUP BY` explicitly. **Cypher has no `GROUP BY` clause at all.**

Grouping is *implicit*:

> **Every non-aggregating expression in `RETURN` (or `WITH`) automatically becomes a grouping key.**

```cypher
RETURN c.city, count(*)
--     |______|  |______|
--     grouping  aggregate
--       key
```

That is exactly equivalent to SQL's `SELECT city, COUNT(*) ... GROUP BY city`.

Add another non-aggregating expression and you have silently added another grouping key:

```cypher
RETURN c.city, c.state, count(*)   -- now grouped by (city, state)
```

> **This is the number one source of "why is my count wrong?" bugs.** If your counts look too
> small and your row count too large, you've accidentally added a grouping key. Read your
> `RETURN` and ask: *which of these are aggregates, and which are keys?*

In [ ]:
# Grouped by ONE key
gdb.run("""
    MATCH (c:Customer)
    RETURN c.segment AS segment, count(*) AS customers
    ORDER BY customers DESC
""")

In [ ]:
# Grouped by TWO keys - notice how the counts fragment
gdb.run("""
    MATCH (c:Customer)
    RETURN c.segment AS segment, c.preferredDevice AS device, count(*) AS customers
    ORDER BY customers DESC
    LIMIT 10
""")

## 5.2 The aggregate functions

| Function | Purpose | Note |
|---|---|---|
| `count(*)` | Count rows | Counts everything, nulls included |
| `count(x)` | Count non-null `x` | Nulls skipped — differs from `count(*)` |
| `count(DISTINCT x)` | Count unique non-null values | |
| `sum` / `avg` / `min` / `max` | The usual numerics | Nulls skipped |
| `stDev` / `stDevP` | Sample / population standard deviation | |
| `percentileCont(x, 0.5)` | Interpolated percentile (median) | |
| `percentileDisc(x, 0.95)` | Nearest-rank percentile | |
| `collect(x)` | **Gather values into a list** | The graph-native one |

`collect()` has no clean SQL equivalent (it resembles `array_agg` / `GROUP_CONCAT`) and it is
enormously useful in graphs — it's how you nest results.

In [ ]:
# Revenue analysis by customer segment
gdb.run("""
    MATCH (c:Customer)-[:PLACED]->(o:Order)
    WHERE o.status = 'Delivered'
    RETURN c.segment                                    AS segment,
           count(DISTINCT c)                            AS customers,
           count(o)                                     AS orders,
           round(sum(o.totalAmount), 2)                 AS revenue,
           round(avg(o.totalAmount), 2)                 AS avgOrderValue,
           round(percentileCont(o.totalAmount, 0.5), 2) AS medianOrderValue,
           max(o.totalAmount)                           AS largestOrder
    ORDER BY revenue DESC
""")

> **Why `count(DISTINCT c)` and not `count(c)`?** After the `MATCH`, we have one row *per order*.
> A customer with 8 orders appears on 8 rows, so `count(c)` returns 8 while
> `count(DISTINCT c)` correctly returns 1. This fan-out effect is constant in graph queries —
> always ask yourself *"what does one row represent at this point?"*

In [ ]:
# collect() builds lists - here, each category with sample products nested inside
gdb.run("""
    MATCH (p:Product)-[:IN_CATEGORY]->(cat:Category)
    WITH cat, collect(p.name)[0..4] AS sampleProducts, count(p) AS productCount
    RETURN cat.name AS category, productCount, sampleProducts
    ORDER BY productCount DESC
    LIMIT 10
""")

## 5.3 `WITH` — the pipeline operator

`WITH` is the most important clause to master after `MATCH`. It **ends one query stage and
begins the next**, passing forward only the variables you name.

Think of it as a Unix pipe: `MATCH | WITH | MATCH | WITH | RETURN`.

You need `WITH` whenever you want to:

1. **Filter on an aggregate** (SQL's `HAVING`)
2. **Aggregate, then keep traversing**
3. **Limit intermediate results before an expensive step**
4. **Compute a value once and reuse it**

### The rule to memorise
> **Anything not named in `WITH` is discarded.** `WITH` defines the entire scope of everything
> downstream. Forgetting to carry a variable forward produces the most common Cypher error you
> will ever see: *"Variable `x` not defined"*.

In [ ]:
# WITH as HAVING: customers who have spent more than 20,000
gdb.run("""
    MATCH (c:Customer)-[:PLACED]->(o:Order)
    WITH c, sum(o.totalAmount) AS totalSpend, count(o) AS orderCount
    WHERE totalSpend > 20000                  // <-- this is HAVING
    RETURN c.name AS customer, c.segment AS segment,
           orderCount, round(totalSpend, 2) AS totalSpend
    ORDER BY totalSpend DESC
    LIMIT 10
""", show_time=True)

> **`WHERE` after `MATCH` filters rows. `WHERE` after `WITH` filters groups.**
> That's the entire difference between SQL's `WHERE` and `HAVING`, expressed with one keyword
> instead of two.

In [ ]:
# Aggregate, THEN keep traversing.
#   Step 1: find the top 5 products by units sold.
#   Step 2: for those products only, go and find their suppliers.
gdb.run("""
    MATCH (p:Product)<-[ci:CONTAINS]-(:Order)
    WITH p, sum(ci.quantity) AS unitsSold
    ORDER BY unitsSold DESC
    LIMIT 5                                    // narrow to 5 products...

    MATCH (s:Supplier)-[sup:SUPPLIES]->(p)     // ...then traverse from them
    RETURN p.name           AS product,
           unitsSold,
           s.name           AS supplier,
           sup.leadTimeDays AS leadDays,
           sup.isPrimary    AS isPrimary
    ORDER BY unitsSold DESC, isPrimary DESC
""")

That query shows the pattern that makes Cypher powerful: **aggregate → narrow → traverse again**.
In SQL you'd need a CTE or subquery; here it's a linear pipeline you read top to bottom.

Also note that `ORDER BY` and `LIMIT` can appear **after `WITH`**, in the middle of a query.
That's a major performance tool — cut the result set to 5 rows *before* doing the expensive
supplier lookup, rather than joining everything and filtering at the end.

In [ ]:
# Multi-stage pipeline: brand performance
gdb.run("""
    // Stage 1 - revenue per brand
    MATCH (p:Product)<-[ci:CONTAINS]-(o:Order)
    WHERE o.status IN ['Delivered', 'Shipped']
    WITH p.brand AS brand,
         sum(ci.totalPrice) AS revenue,
         count(DISTINCT o)  AS orders,
         count(DISTINCT p)  AS products

    // Stage 2 - keep only brands with meaningful volume
    WHERE orders >= 20

    // Stage 3 - derive a metric and rank
    WITH brand, revenue, orders, products,
         round(revenue / orders, 2) AS revenuePerOrder
    ORDER BY revenue DESC
    LIMIT 10

    RETURN brand, products, orders, round(revenue, 2) AS revenue, revenuePerOrder
""")

## 5.4 Aggregating over the graph structure itself

Graph databases let you aggregate on something relational databases express only awkwardly:
**how connected a node is**. This is called **degree**.

In [ ]:
# Degree centrality: which products sit at the centre of the purchase graph?
gdb.run("""
    MATCH (p:Product)
    RETURN p.name  AS product,
           p.brand AS brand,
           count { (p)<-[:CONTAINS]-(:Order) }    AS timesOrdered,
           count { (p)<-[:REVIEWS]-(:Review) }    AS reviewCount,
           count { (p)<-[:SUPPLIES]-(:Supplier) } AS supplierCount,
           count { (p)<-[:STOCKS]-(:Warehouse) }  AS warehouseCount
    ORDER BY timesOrdered DESC
    LIMIT 10
""", show_time=True)

### The `count { ... }` subquery

`count { (p)<-[:CONTAINS]-(:Order) }` is a **count subquery** (Neo4j 5+). It counts matches of a
pattern *without* fanning out your rows.

Compare the alternative:

```cypher
MATCH (p:Product)<-[:CONTAINS]-(o:Order)     -- creates one row per order!
RETURN p.name, count(o)                      -- then collapses them again
```

That version fans out to 14,456 rows and collapses back to 1,000. The `count {}` form keeps
1,000 rows throughout and evaluates each count in place. It's clearer, usually faster — and
crucially it **keeps products with zero orders**, which the `MATCH` version silently drops.

There are siblings: `exists { ... }`, and in Neo4j 5.23+ also `collect { ... }`.

In [ ]:
# Time-series aggregation using temporal components
gdb.run("""
    MATCH (o:Order)
    WHERE o.status <> 'Cancelled'
    RETURN o.orderDate.year    AS year,
           o.orderDate.quarter AS quarter,
           count(*)            AS orders,
           round(sum(o.totalAmount), 2) AS revenue,
           round(avg(o.totalAmount), 2) AS avgOrder
    ORDER BY year, quarter
""")

In [ ]:
# Visualise that trend
import matplotlib.pyplot as plt

trend = gdb.run("""
    MATCH (o:Order)
    WHERE o.status <> 'Cancelled'
    RETURN o.orderDate.year AS year, o.orderDate.month AS month,
           count(*) AS orders, sum(o.totalAmount) AS revenue
    ORDER BY year, month
""")
trend["period"] = (trend["year"].astype(str) + "-"
                   + trend["month"].astype(str).str.zfill(2))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
ax1.bar(trend["period"], trend["orders"], color="#4C9BE8")
ax1.set_ylabel("Orders")
ax1.set_title("Order volume and revenue by month")
ax2.plot(trend["period"], trend["revenue"] / 1000, marker="o", color="#E86A6A")
ax2.set_ylabel("Revenue (thousands)")
ax2.set_xlabel("Month")
plt.xticks(rotation=70, fontsize=7)
ax1.grid(axis="y", alpha=0.3)
ax2.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 5.5 Exercises — Part 5

**Exercise 5.1** — For each **city**, compute the number of customers, total lifetime revenue and
average loyalty points. Show only cities with at least 3 customers, ordered by revenue.

<details>
<summary><b>&#128161; Show solution &mdash; 5.1</b></summary>

```cypher
MATCH (c:Customer)
WITH c.city AS city,
     count(*)               AS customers,
     sum(c.lifetimeRevenue) AS revenue,
     avg(c.loyaltyPoints)   AS avgPoints
WHERE customers >= 3
RETURN city, customers,
       round(revenue, 2) AS revenue,
       round(avgPoints)  AS avgLoyaltyPoints
ORDER BY revenue DESC
LIMIT 20
```
The `WHERE` sits after `WITH`, so it filters *groups*, not rows — this is `HAVING`.

</details>

**Exercise 5.2** — Find the top 10 categories by **total revenue**, where revenue is the sum of
`CONTAINS.totalPrice` for delivered orders. Also show how many distinct products and customers
contributed.

<details>
<summary><b>&#128161; Show solution &mdash; 5.2</b></summary>

```cypher
MATCH (c:Customer)-[:PLACED]->(o:Order)-[ci:CONTAINS]->(p:Product)-[:IN_CATEGORY]->(cat:Category)
WHERE o.status = 'Delivered'
RETURN cat.name          AS category,
       count(DISTINCT p) AS products,
       count(DISTINCT c) AS customers,
       count(DISTINCT o) AS orders,
       round(sum(ci.totalPrice), 2) AS revenue
ORDER BY revenue DESC
LIMIT 10
```
Every `count` needs `DISTINCT` because the `MATCH` produces one row per *order line*, so
products, customers and orders all repeat.

</details>

**Exercise 5.3** — For each product, calculate its **actual** average rating from the `Review`
nodes and compare it to the `avgRating` property stored on the product. List the 10 products
where the two differ most.

<details>
<summary><b>&#128161; Show solution &mdash; 5.3</b></summary>

```cypher
MATCH (r:Review)-[:REVIEWS]->(p:Product)
WITH p, avg(r.rating) AS computedRating, count(r) AS reviewsInGraph
WHERE reviewsInGraph >= 3
RETURN p.name                                      AS product,
       reviewsInGraph,
       round(computedRating, 2)                    AS computed,
       p.avgRating                                 AS stored,
       round(abs(computedRating - p.avgRating), 2) AS difference
ORDER BY difference DESC
LIMIT 10
```
**Why they differ:** `p.avgRating` was computed over the *full* review set in the source system,
whereas our graph holds only reviews written by our 1,201 sampled customers. This is a realistic
and valuable lesson — **denormalised aggregates go stale**, and the graph lets you recompute truth.

</details>

**Exercise 5.4** *(harder)* — Which **suppliers** carry the highest "risk concentration"?
Define it as: the supplier supplies products that together generated the most revenue, *and* the
supplier is the **only** supplier for those products. Return supplier, product count,
sole-sourced product count, and revenue at risk.

<details>
<summary><b>&#128161; Show solution &mdash; 5.4</b></summary>

```cypher
// Step 1: how many suppliers does each product have?
MATCH (p:Product)
WITH p, count { (p)<-[:SUPPLIES]-(:Supplier) } AS supplierCount

// Step 2: revenue per product
MATCH (p)<-[ci:CONTAINS]-(o:Order)
WHERE o.status IN ['Delivered', 'Shipped']
WITH p, supplierCount, sum(ci.totalPrice) AS productRevenue

// Step 3: attribute back to suppliers
MATCH (s:Supplier)-[:SUPPLIES]->(p)
WITH s,
     count(p)                                                       AS products,
     sum(CASE WHEN supplierCount = 1 THEN 1 ELSE 0 END)             AS soleSourced,
     sum(CASE WHEN supplierCount = 1 THEN productRevenue ELSE 0 END) AS revenueAtRisk
WHERE soleSourced > 0
RETURN s.name AS supplier, products, soleSourced,
       round(revenueAtRisk, 2) AS revenueAtRisk
ORDER BY revenueAtRisk DESC
LIMIT 10
```
A genuinely useful supply-chain query, and it exercises the full toolkit: a `count {}` subquery,
multi-stage `WITH`, and `CASE` inside an aggregate.

</details>

## 5.6 Checkpoint 5

In [ ]:
seg = gdb.run("MATCH (c:Customer)-[:PLACED]->(o:Order) "
              "RETURN c.segment AS s, count(DISTINCT c) AS n, "
              "sum(o.totalAmount) AS rev ORDER BY rev DESC")
having = gdb.run("MATCH (c:Customer)-[:PLACED]->(o:Order) "
                 "WITH c, sum(o.totalAmount) AS t WHERE t > 20000 RETURN count(c) AS n")
deg = gdb.run("MATCH (p:Product) RETURN p.name AS n, "
              "count { (p)<-[:CONTAINS]-(:Order) } AS d ORDER BY d DESC LIMIT 5")

checkpoint("Part 5 - Aggregation", [
    ("Segment aggregation returns rows", len(seg) > 0),
    ("HAVING-style filter works",        int(having.iloc[0]["n"]) > 0),
    ("count{} subquery works",           len(deg) == 5 and int(deg.iloc[0]["d"]) > 0),
    ("I understand implicit grouping",   True),
])

---
# Part 6 — Traversal and Paths

This is the part where graph databases do something relational databases genuinely cannot do
comfortably. Everything so far had a reasonable SQL equivalent. What follows does not.

## 6.1 Fixed-length multi-hop traversal

We've already done two hops. Let's go deeper and build the classic recommendation.

In [ ]:
# SIX HOPS: "customers who bought what you bought, also bought..."
# This is the exact query we wrote in 18 lines of SQL back in Part 1.
recs = gdb.run("""
    MATCH (me:Customer {customerId: $cid})-[:PLACED]->(:Order)-[:CONTAINS]->(bought:Product)
    MATCH (bought)<-[:CONTAINS]-(:Order)<-[:PLACED]-(peer:Customer)
    WHERE peer <> me
    MATCH (peer)-[:PLACED]->(:Order)-[:CONTAINS]->(rec:Product)
    WHERE NOT (me)-[:PLACED]->(:Order)-[:CONTAINS]->(rec)
    RETURN rec.name  AS recommendation,
           rec.brand AS brand,
           rec.price AS price,
           count(DISTINCT peer) AS peersWhoBought
    ORDER BY peersWhoBought DESC, price DESC
    LIMIT 10
""", {"cid": CID}, show_time=True)
recs

### Read that query again

Seven lines. Zero join conditions. Compare it to the SQL in §1.4 — and notice that the Cypher
version is the one you could explain to a product manager.

Three details worth naming:

- **`peer <> me`** excludes the customer from their own recommendations. Node comparison
  compares *identity*, not properties.
- **`WHERE NOT (me)-[:PLACED]->(:Order)-[:CONTAINS]->(rec)`** — a negated **path** filter, three
  hops long, used as a boolean. This is the clause that removes things they already own.
- **`count(DISTINCT peer)`** is the recommendation's *support*: how many similar customers back it.

## 6.2 Variable-length paths

Sometimes you don't know how deep to go. Our category tree is the perfect example: some products
sit at level 2, others at level 3, so the distance to the root **varies by product**.

```cypher
-[:PARENT_OF*1..3]->    // between 1 and 3 hops
-[:PARENT_OF*2]->       // exactly 2 hops
-[:PARENT_OF*]->        // 1 or more - DANGEROUS, always bound it
-[:PARENT_OF*0..2]->    // 0 hops means "or the node itself"
```

> **Always put an upper bound on variable-length patterns.** `*` with no bound on a densely
> connected graph will explore an exponentially growing frontier and can hang your database.
> `*1..4` is almost always enough, and it is always safer.

In [ ]:
# Where does each category sit in the tree, and what is its root?
gdb.run("""
    MATCH path = (root:Category)-[:PARENT_OF*0..2]->(cat:Category)
    WHERE NOT (:Category)-[:PARENT_OF]->(root)      // root = has no parent
    RETURN root.name          AS rootCategory,
           cat.name           AS category,
           cat.level          AS level,
           length(path)       AS hopsFromRoot
    ORDER BY rootCategory, hopsFromRoot, category
    LIMIT 20
""")

Note the two idioms in that query:

- **`*0..2`** — the `0` makes the path optional, so a root category matches itself with
  `hopsFromRoot = 0`. Without the zero, roots would vanish from the results.
- **`WHERE NOT (:Category)-[:PARENT_OF]->(root)`** — "has no incoming parent", which is how you
  identify tree roots without storing a flag.

In [ ]:
# ROLL-UP: total revenue per TOP-LEVEL category.
# Products attach at level 2 or 3, so the number of hops to the root varies.
# A variable-length path handles both cases in a single query - the thing that
# would need a recursive CTE in SQL.
gdb.run("""
    MATCH (o:Order)-[ci:CONTAINS]->(p:Product)-[:IN_CATEGORY]->(leaf:Category)
    WHERE o.status IN ['Delivered', 'Shipped']
    MATCH (root:Category)-[:PARENT_OF*0..2]->(leaf)
    WHERE root.level = 1
    RETURN root.name                   AS topLevelCategory,
           count(DISTINCT p)           AS products,
           count(DISTINCT o)           AS orders,
           round(sum(ci.totalPrice), 2) AS revenue
    ORDER BY revenue DESC
""", show_time=True)

> **Stop and appreciate this one.** In SQL, rolling a variable-depth hierarchy up to its root
> requires a `WITH RECURSIVE` common table expression — typically 15 lines, and notoriously
> hard to read and debug. Here it's `-[:PARENT_OF*0..2]->`.
>
> Hierarchies (org charts, bills of materials, category trees, account structures, network
> topology) are the single most common reason teams move a workload to a graph database.

In [ ]:
# Draw the full path from a product up to its root category
draw("""
    MATCH path = (root:Category)-[:PARENT_OF*0..2]->(:Category)<-[:IN_CATEGORY]-(p:Product)
    WHERE root.level = 1 AND root.name = 'Electronics'
    WITH path LIMIT 18
    RETURN path
""", title="Products rolled up through the category tree to 'Electronics'", figsize=(14, 9))

## 6.3 Shortest path

`shortestPath()` finds the **shortest** connection between two nodes. Neo4j implements it with a
bidirectional breadth-first search — it expands from both ends simultaneously and meets in the
middle, which is dramatically faster than searching from one side.

This is the query behind "degrees of separation", fraud-ring proximity, network blast radius, and
supply-chain trace-back.

In [ ]:
# How are two customers connected through the products they bought?
pair = gdb.run("""
    MATCH (a:Customer)-[:PLACED]->(:Order)-[:CONTAINS]->(p:Product)
          <-[:CONTAINS]-(:Order)<-[:PLACED]-(b:Customer)
    WHERE a.customerId < b.customerId
    WITH a, b, count(DISTINCT p) AS shared
    WHERE shared >= 2
    RETURN a.customerId AS a, b.customerId AS b, shared
    ORDER BY shared DESC LIMIT 1
""")
A, B = int(pair.iloc[0]["a"]), int(pair.iloc[0]["b"])
print("Tracing the connection between customer {} and customer {}\n".format(A, B))

gdb.run("""
    MATCH (a:Customer {customerId: $a}), (b:Customer {customerId: $b})
    MATCH path = shortestPath((a)-[:PLACED|CONTAINS*..8]-(b))
    RETURN length(path)                          AS hops,
           [n IN nodes(path) |
              coalesce(n.name, toString(n.orderId))] AS route,
           [r IN relationships(path) | type(r)]  AS relationshipTypes
""", {"a": A, "b": B})

In [ ]:
# See it
draw("""
    MATCH (a:Customer {customerId: $a}), (b:Customer {customerId: $b})
    MATCH path = shortestPath((a)-[:PLACED|CONTAINS*..8]-(b))
    RETURN path
""", {"a": A, "b": B},
     title="Shortest path between two customers", figsize=(12, 5))

### `shortestPath` rules you must know

| Rule | Detail |
|---|---|
| Both endpoints must be **bound** | Match them first, in a separate `MATCH` or via `WITH` |
| Always **bound the hops** | `*..8`, never `*` — unbounded is a performance trap |
| Use **undirected** patterns for reachability | `-[:R*..8]-` unless direction genuinely matters |
| Multiple types are allowed | `[:PLACED\|CONTAINS*..8]` — a pipe-separated list |
| `allShortestPaths()` returns **all** ties | Same length, several routes |

> **Common error:** *"shortestPath(...) requires bound nodes"*. It means you tried to find a
> path from a pattern that Neo4j hasn't resolved to concrete nodes yet. Fix: match the endpoints
> in their own `MATCH` clause first, exactly as we did above.

In [ ]:
# allShortestPaths: every equally-short route between two products
gdb.run("""
    MATCH (p1:Product {productId: 2001}), (p2:Product)
    WHERE p2.productId = 2050
    MATCH path = allShortestPaths((p1)-[:CONTAINS|IN_CATEGORY|SUPPLIES*..6]-(p2))
    RETURN length(path) AS hops,
           [n IN nodes(path) | coalesce(n.name, labels(n)[0])] AS route
    LIMIT 5
""")

## 6.4 Working with paths as values

A path is a **first-class value** in Cypher. Bind it with `path = (...)` and you get a whole
toolkit:

| Function | Returns |
|---|---|
| `length(path)` | Number of **relationships** (not nodes) |
| `nodes(path)` | List of nodes, in order |
| `relationships(path)` | List of relationships, in order |
| `[n IN nodes(path) \| n.name]` | List comprehension over the path |

In [ ]:
# Path analysis: how many distinct suppliers sit behind each order?
gdb.run("""
    MATCH path = (o:Order)-[:CONTAINS]->(:Product)<-[:SUPPLIES]-(s:Supplier)
    WITH o, collect(DISTINCT s.name) AS suppliers, count(DISTINCT s) AS supplierCount
    WHERE supplierCount >= 4
    RETURN o.orderId     AS orderId,
           o.totalAmount AS amount,
           supplierCount,
           suppliers[0..4] AS sampleSuppliers
    ORDER BY supplierCount DESC
    LIMIT 10
""")

## 6.5 Exercises — Part 6

**Exercise 6.1** — Find every category that is a **leaf** (has no children) and report how many
products sit in it. Order by product count.

<details>
<summary><b>&#128161; Show solution &mdash; 6.1</b></summary>

```cypher
MATCH (cat:Category)
WHERE NOT (cat)-[:PARENT_OF]->(:Category)
RETURN cat.name  AS leafCategory,
       cat.level AS level,
       count { (cat)<-[:IN_CATEGORY]-(:Product) } AS products
ORDER BY products DESC
LIMIT 20
```
The leaf test is a negated pattern; `count {}` avoids fanning out rows and keeps empty
categories visible (they'd show `0`).

</details>

**Exercise 6.2** — For the customer stored in `CID`, find all customers within **4 hops** through
any of `PLACED` or `CONTAINS`, and count how many there are at each distance.

<details>
<summary><b>&#128161; Show solution &mdash; 6.2</b></summary>

```cypher
MATCH (me:Customer {customerId: $cid})
MATCH path = (me)-[:PLACED|CONTAINS*1..4]-(other:Customer)
WHERE other <> me
WITH other, min(length(path)) AS distance
RETURN distance, count(other) AS customers
ORDER BY distance
```
`min(length(path))` matters: there are many routes to the same customer, and you want each
customer counted once, at their *shortest* distance. Without the `min`, one customer would be
counted at several distances.

Run it with `{"cid": CID}` as parameters.

</details>

**Exercise 6.3** — Build a "category breadcrumb" for each product: a list showing the full path
from the root category down to the product's own category, e.g.
`['Electronics', 'Smartphones', 'Android Phones']`.

<details>
<summary><b>&#128161; Show solution &mdash; 6.3</b></summary>

```cypher
MATCH (p:Product)-[:IN_CATEGORY]->(leaf:Category)
MATCH path = (root:Category)-[:PARENT_OF*0..2]->(leaf)
WHERE root.level = 1
RETURN p.name                              AS product,
       [c IN nodes(path) | c.name]         AS breadcrumb,
       length(path)                        AS depth
ORDER BY depth DESC, product
LIMIT 15
```
`nodes(path)` returns the nodes **in traversal order**, which is exactly what a breadcrumb needs.
This single query replaces what would be a recursive CTE plus string aggregation in SQL.

</details>

**Exercise 6.4** *(harder)* — **Supply-chain trace-back.** A supplier has shipped a defective
batch. Given a supplier id, find every customer who could be affected: those who ordered a
product supplied by that supplier, where the order was `Delivered`. Return customer name, email,
the affected product, and the order date.

<details>
<summary><b>&#128161; Show solution &mdash; 6.4</b></summary>

```cypher
MATCH (s:Supplier {supplierId: $sid})-[:SUPPLIES]->(p:Product)
MATCH (p)<-[:CONTAINS]-(o:Order)<-[:PLACED]-(c:Customer)
WHERE o.status = 'Delivered'
RETURN c.name      AS customer,
       c.email     AS email,
       p.name      AS affectedProduct,
       o.orderId   AS orderId,
       o.orderDate AS orderDate
ORDER BY o.orderDate DESC
LIMIT 25
```

Pick a supplier with plenty of products first:

```cypher
MATCH (s:Supplier)-[:SUPPLIES]->(p:Product)
RETURN s.supplierId, s.name, count(p) AS products
ORDER BY products DESC LIMIT 5
```

**This is the query that sells graph databases to regulated industries.** Product recalls,
pharmaceutical batch tracing, food-safety contamination tracking and financial exposure analysis
are all the same shape — and they all need it answered in seconds, not overnight.

</details>

## 6.6 Checkpoint 6

In [ ]:
h = gdb.run("""
    MATCH (o:Order)-[ci:CONTAINS]->(p:Product)-[:IN_CATEGORY]->(leaf:Category)
    MATCH (root:Category)-[:PARENT_OF*0..2]->(leaf)
    WHERE root.level = 1
    RETURN root.name AS c, round(sum(ci.totalPrice),2) AS rev
    ORDER BY rev DESC
""")
sp = gdb.run("""
    MATCH (a:Customer {customerId:$a}), (b:Customer {customerId:$b})
    MATCH path = shortestPath((a)-[:PLACED|CONTAINS*..8]-(b))
    RETURN length(path) AS hops
""", {"a": A, "b": B})

checkpoint("Part 6 - Traversal and paths", [
    ("6-hop recommendation returns rows",   len(recs) > 0),
    ("Hierarchy roll-up covers 10 roots",   len(h) >= 8),
    ("shortestPath found a route",          len(sp) > 0 and int(sp.iloc[0]["hops"]) > 0),
    ("I know to always bound *n..m",        True),
])

---
# Part 7 — Advanced Cypher

## 7.1 `OPTIONAL MATCH` — the LEFT JOIN of Cypher

`MATCH` drops rows that don't match. `OPTIONAL MATCH` keeps them and fills the missing
variables with `null`. It behaves exactly like SQL's `LEFT OUTER JOIN`.

In [ ]:
# WRONG: this silently drops every product that has no reviews
wrong = gdb.run("""
    MATCH (p:Product)-[:IN_CATEGORY]->(:Category {name: 'Running Shoes'})
    MATCH (p)<-[:REVIEWS]-(r:Review)
    RETURN count(DISTINCT p) AS productsReturned
""")

# RIGHT: OPTIONAL MATCH keeps unreviewed products, with null reviews
right = gdb.run("""
    MATCH (p:Product)-[:IN_CATEGORY]->(:Category {name: 'Running Shoes'})
    OPTIONAL MATCH (p)<-[:REVIEWS]-(r:Review)
    RETURN count(DISTINCT p) AS productsReturned
""")

print("Plain MATCH    :", int(wrong.iloc[0]["productsReturned"]), "products")
print("OPTIONAL MATCH :", int(right.iloc[0]["productsReturned"]), "products")
print("\nThe difference is the products with zero reviews - the ones you most")
print("often actually care about.")

In [ ]:
# A realistic product health report using OPTIONAL MATCH
gdb.run("""
    MATCH (p:Product)-[:IN_CATEGORY]->(cat:Category)
    OPTIONAL MATCH (p)<-[:REVIEWS]-(r:Review)
    OPTIONAL MATCH (p)<-[ci:CONTAINS]-(o:Order)
    OPTIONAL MATCH (p)<-[:SUPPLIES]-(s:Supplier)
    WITH p, cat,
         count(DISTINCT r)  AS reviews,
         avg(r.rating)      AS avgRating,
         count(DISTINCT o)  AS orders,
         sum(ci.totalPrice) AS revenue,
         count(DISTINCT s)  AS suppliers
    RETURN p.name                        AS product,
           cat.name                      AS category,
           reviews,
           round(coalesce(avgRating, 0), 2) AS avgRating,
           orders,
           round(coalesce(revenue, 0), 2)   AS revenue,
           suppliers
    ORDER BY revenue DESC
    LIMIT 12
""")

> **Notice `coalesce(revenue, 0)`.** `sum()` over an empty set returns `0`, but `avg()` over an
> empty set returns `null`. Mixing them without `coalesce` gives you a column with holes in it.
> Whenever you use `OPTIONAL MATCH`, plan for the nulls it will produce.

## 7.2 `CASE` — conditional logic

Two forms, matching SQL exactly.

In [ ]:
gdb.run("""
    MATCH (c:Customer)
    OPTIONAL MATCH (c)-[:PLACED]->(o:Order)
    WITH c, count(o) AS orders, coalesce(sum(o.totalAmount), 0) AS spend
    RETURN
      // "simple" CASE - compare one expression against values
      CASE c.segment
        WHEN 'VIP'     THEN 'Priority support'
        WHEN 'At-Risk' THEN 'Win-back campaign'
        WHEN 'Churned' THEN 'Reactivation offer'
        ELSE 'Standard'
      END AS treatment,

      // "generic" CASE - arbitrary boolean conditions, evaluated in order
      CASE
        WHEN spend > 30000 THEN 'Platinum'
        WHEN spend > 15000 THEN 'Gold'
        WHEN spend >  5000 THEN 'Silver'
        WHEN orders = 0    THEN 'Never purchased'
        ELSE 'Bronze'
      END AS valueTier,

      count(*)                AS customers,
      round(avg(spend), 2)    AS avgSpend
    ORDER BY customers DESC
""")

## 7.3 List comprehensions and pattern comprehensions

Cypher borrows list comprehensions from Python, and then adds a graph-native twist.

```cypher
[x IN list WHERE condition | expression]     -- list comprehension
[(a)-[:R]->(b) WHERE cond | b.name]          -- PATTERN comprehension
```

A **pattern comprehension** runs a mini-pattern-match inside an expression and collects the
results into a list — without fanning out your rows. It's `collect()` plus `MATCH` in one step,
and it's the single most elegant thing in the language.

In [ ]:
# Build a nested customer profile in ONE query, with no row explosion.
gdb.run("""
    MATCH (c:Customer)
    WHERE c.segment = 'VIP'
    WITH c LIMIT 5
    RETURN c.name AS customer,

           // pattern comprehension: products this customer bought
           [(c)-[:PLACED]->(:Order)-[:CONTAINS]->(p:Product) | p.name][0..5]
              AS someProducts,

           // with a filter inside the pattern
           [(c)-[:PLACED]->(o:Order) WHERE o.status = 'Returned' | o.orderId]
              AS returnedOrders,

           // reviews they wrote, only the critical ones
           [(c)-[:WROTE]->(r:Review) WHERE r.rating <= 2 | r.title]
              AS negativeReviews,

           // a plain list comprehension over a list property
           [(c)-[:PLACED]->(:Order)-[:CONTAINS]->(p:Product) | p.price]
              AS pricesPaid
""")

### Why pattern comprehensions matter

Without them, building that profile means either several `OPTIONAL MATCH` clauses with careful
`collect(DISTINCT ...)` (and the fan-out bugs that come with it), or several separate queries.

With them, each column is independent, self-contained, and readable. This is how you build a
**nested JSON API response** from a graph in a single round-trip.

In [ ]:
# reduce() folds a list into a single value - like Python's functools.reduce
gdb.run("""
    MATCH (o:Order)-[ci:CONTAINS]->(:Product)
    WITH o, collect(ci.totalPrice) AS lineTotals
    WHERE size(lineTotals) >= 3
    RETURN o.orderId AS orderId,
           size(lineTotals) AS lines,
           lineTotals,
           reduce(total = 0.0, x IN lineTotals | total + x) AS computedTotal,
           round(o.totalAmount, 2) AS storedTotal
    LIMIT 8
""")

### The list function toolbox

| Function | Purpose |
|---|---|
| `size(list)` | Length |
| `head(list)` / `last(list)` | First / last element |
| `list[0..3]` | Slice (Python-style, end-exclusive) |
| `range(1, 10)` | Generate a list |
| `reverse(list)` | Reverse |
| `reduce(acc = 0, x IN list \| acc + x)` | Fold |
| `[x IN list WHERE ...]` | Filter |
| `any/all/none/single(x IN list WHERE ...)` | Predicates returning a boolean |
| `apoc.coll.*` | Richer set operations — **available on Aura**, APOC Core is preinstalled |

In [ ]:
# Predicate functions over the tags list property we built during the load
gdb.run("""
    MATCH (p:Product)
    WHERE size(p.tags) >= 2
      AND any(t IN p.tags WHERE t CONTAINS 'arch-support')
    RETURN p.name AS product, p.tags AS tags, p.price AS price
    LIMIT 10
""")

## 7.4 Subqueries with `CALL { ... }`

A `CALL {}` subquery runs an independent query **for each incoming row** and returns rows back
into the outer query. It's the tool for "top N per group", which is famously awkward in SQL.

```cypher
CALL (outerVar) {          -- Neo4j 5.23+ scope syntax
    ...
    RETURN something
}
```

On Neo4j versions before 5.23 the equivalent is `CALL { WITH outerVar ... }`. Both are shown
below; use whichever your Aura instance accepts.

In [ ]:
# TOP 3 PRODUCTS PER CATEGORY - the classic "top N per group" problem.
try:
    result = gdb.run("""
        MATCH (cat:Category)
        WHERE cat.level = 1
        CALL (cat) {
            MATCH (cat)-[:PARENT_OF*0..2]->(:Category)<-[:IN_CATEGORY]-(p:Product)
            MATCH (p)<-[ci:CONTAINS]-(o:Order)
            WHERE o.status = 'Delivered'
            RETURN p.name AS product, sum(ci.totalPrice) AS revenue
            ORDER BY revenue DESC
            LIMIT 3
        }
        RETURN cat.name AS category, product, round(revenue, 2) AS revenue
        ORDER BY category, revenue DESC
    """)
except Exception:
    # Fallback for Neo4j < 5.23
    result = gdb.run("""
        MATCH (cat:Category)
        WHERE cat.level = 1
        CALL {
            WITH cat
            MATCH (cat)-[:PARENT_OF*0..2]->(:Category)<-[:IN_CATEGORY]-(p:Product)
            MATCH (p)<-[ci:CONTAINS]-(o:Order)
            WHERE o.status = 'Delivered'
            RETURN p.name AS product, sum(ci.totalPrice) AS revenue
            ORDER BY revenue DESC
            LIMIT 3
        }
        RETURN cat.name AS category, product, round(revenue, 2) AS revenue
        ORDER BY category, revenue DESC
    """)
result

> **Why this is hard in SQL:** you need a window function
> (`ROW_NUMBER() OVER (PARTITION BY category ORDER BY revenue DESC)`) wrapped in a subquery to
> filter `WHERE rn <= 3`. The `CALL {}` version reads like what it does: *for each category, run
> this query and take the top 3.*

In [ ]:
# EXISTS subquery - a full pattern (with WHERE) used as a boolean
gdb.run("""
    MATCH (c:Customer)
    WHERE EXISTS {
        MATCH (c)-[:PLACED]->(o:Order)-[:CONTAINS]->(p:Product)
        WHERE p.price > 1200 AND o.status = 'Delivered'
    }
    AND NOT EXISTS {
        MATCH (c)-[:WROTE]->(r:Review)
        WHERE r.rating <= 2
    }
    RETURN c.name AS customer, c.segment AS segment,
           c.lifetimeRevenue AS revenue
    ORDER BY revenue DESC
    LIMIT 10
""")

`EXISTS {}` is more powerful than a bare pattern in `WHERE`, because it can contain its own
`WHERE`, several `MATCH` clauses, and further filtering. Use the simple pattern form
(`WHERE (c)-[:WROTE]->(:Review)`) when the test is a single hop; reach for `EXISTS {}` when the
test needs conditions.

## 7.5 `UNION` — combining result sets

In [ ]:
# UNION requires identical column names in every branch.
# UNION removes duplicates; UNION ALL keeps them (and is faster).
gdb.run("""
    MATCH (c:Customer)
    WHERE c.lifetimeRevenue > 40000
    RETURN c.name AS name, 'High spender' AS reason, c.lifetimeRevenue AS metric

    UNION

    MATCH (c:Customer)-[:PLACED]->(o:Order)
    WHERE o.status = 'Returned'
    WITH c, count(o) AS returns
    WHERE returns >= 3
    RETURN c.name AS name, 'Serial returner' AS reason, toFloat(returns) AS metric

    UNION

    MATCH (c:Customer)-[:WROTE]->(r:Review)
    WITH c, avg(r.rating) AS rating, count(r) AS n
    WHERE n >= 3 AND rating <= 2.0
    RETURN c.name AS name, 'Persistent critic' AS reason, round(rating, 2) AS metric
""")

> **The type trap:** every branch of a `UNION` must return the *same column names*, and it's good
> practice to keep the *types* consistent too. Notice `toFloat(returns)` in branch two — without
> it, one branch returns an integer where the others return floats, which makes the resulting
> DataFrame column awkward to work with.

## 7.6 Exercises — Part 7

**Exercise 7.1** — Build a "customer 360" for the 5 highest-spending customers. Using **pattern
comprehensions**, one row per customer, include: their 3 most recent order ids, all distinct
brands they bought, the categories they shopped, and the count of reviews they wrote.

<details>
<summary><b>&#128161; Show solution &mdash; 7.1</b></summary>

```cypher
MATCH (c:Customer)-[:PLACED]->(o:Order)
WITH c, sum(o.totalAmount) AS spend
ORDER BY spend DESC
LIMIT 5
RETURN c.name AS customer,
       round(spend, 2) AS totalSpend,
       [(c)-[:PLACED]->(o:Order) | o.orderId][0..3]                 AS recentOrders,
       [(c)-[:PLACED]->(:Order)-[:CONTAINS]->(p:Product) | p.brand] AS brands,
       [(c)-[:PLACED]->(:Order)-[:CONTAINS]->(:Product)
            -[:IN_CATEGORY]->(cat:Category) | cat.name]             AS categories,
       count { (c)-[:WROTE]->(:Review) }                            AS reviewsWritten
```
To de-duplicate the brand and category lists, wrap them:
`apoc.coll.toSet([...])` (APOC Core is available on Aura), or in pure Cypher use a
`WITH ... collect(DISTINCT ...)` stage instead of a comprehension.

</details>

**Exercise 7.2** — For every warehouse, list the top 3 products by units shipped, using
`CALL {}`.

<details>
<summary><b>&#128161; Show solution &mdash; 7.2</b></summary>

```cypher
MATCH (w:Warehouse)
CALL (w) {
    MATCH (w)<-[:SHIPPED_FROM]-(o:Order)-[ci:CONTAINS]->(p:Product)
    RETURN p.name AS product, sum(ci.quantity) AS units
    ORDER BY units DESC
    LIMIT 3
}
RETURN w.name AS warehouse, product, units
ORDER BY warehouse, units DESC
```
(Use the `CALL { WITH w ... }` form if your Aura instance predates 5.23.)

Note the traversal: we go *from* the warehouse *back* through `SHIPPED_FROM` to the order, then
forward into its line items. Direction matters here, and reading the arrows carefully is the
whole skill.

</details>

**Exercise 7.3** *(harder)* — Classify every product into a lifecycle stage using `CASE`:
`'Star'` (high revenue, high rating), `'Cash cow'` (high revenue, low rating), `'Question mark'`
(low revenue, high rating), `'Dog'` (low revenue, low rating), `'Unsold'` (never ordered).
Use the median revenue as the high/low threshold. Report the count in each stage.

<details>
<summary><b>&#128161; Show solution &mdash; 7.3</b></summary>

```cypher
MATCH (p:Product)
OPTIONAL MATCH (p)<-[ci:CONTAINS]-(o:Order)
WHERE o.status IN ['Delivered', 'Shipped']
WITH p, coalesce(sum(ci.totalPrice), 0) AS revenue
WITH collect([p.productId, revenue, p.avgRating]) AS all,
     percentileCont(revenue, 0.5) AS medianRevenue
UNWIND all AS row
WITH row[1] AS revenue, row[2] AS rating, medianRevenue
RETURN CASE
         WHEN revenue = 0                                    THEN 'Unsold'
         WHEN revenue >= medianRevenue AND rating >= 4.0     THEN 'Star'
         WHEN revenue >= medianRevenue AND rating <  4.0     THEN 'Cash cow'
         WHEN revenue <  medianRevenue AND rating >= 4.0     THEN 'Question mark'
         ELSE 'Dog'
       END AS stage,
       count(*) AS products
ORDER BY products DESC
```
The teaching point is the **`collect` / `UNWIND` sandwich**: you must aggregate over the whole
set to compute the median, but you then need the individual rows back to classify them.
`collect(...)` folds everything into one row so the aggregate can be computed, and `UNWIND`
unfolds it again with that aggregate now available on every row. Memorise this pattern — it's
how you do any "compare each row to a global statistic" calculation in Cypher.

</details>

## 7.7 Checkpoint 7

In [ ]:
o1 = gdb.run("""
    MATCH (p:Product)-[:IN_CATEGORY]->(:Category {name:'Running Shoes'})
    OPTIONAL MATCH (p)<-[:REVIEWS]-(r:Review)
    RETURN count(DISTINCT p) AS n
""")
pc = gdb.run("""
    MATCH (c:Customer) WITH c LIMIT 3
    RETURN c.name AS n,
           [(c)-[:PLACED]->(:Order)-[:CONTAINS]->(p:Product) | p.name][0..3] AS prods
""")
ex = gdb.run("""
    MATCH (c:Customer)
    WHERE EXISTS { MATCH (c)-[:PLACED]->(:Order)-[:CONTAINS]->(p:Product)
                   WHERE p.price > 1200 }
    RETURN count(c) AS n
""")

checkpoint("Part 7 - Advanced Cypher", [
    ("OPTIONAL MATCH keeps unmatched rows", int(o1.iloc[0]["n"]) > 0),
    ("Pattern comprehension works",         len(pc) == 3),
    ("EXISTS subquery works",               int(ex.iloc[0]["n"]) > 0),
    ("CALL {} top-N-per-group works",       len(result) > 0),
])

---
# Part 8 — Graph Analytics in Pure Cypher

Neo4j has a **Graph Data Science (GDS)** library with PageRank, Louvain community detection,
node embeddings and more. **GDS is not available on Aura Free.**

That's a blessing for learning. In this part we implement the core algorithms **by hand in
Cypher**, which means you'll understand what the library does rather than just calling it.

## 8.1 First, let's extend the graph

Fraud analysis needs payment data, which we deliberately left out of the initial load. Adding a
new entity to a live graph is a routine, realistic task — so let's do it properly.

In [ ]:
# Add Payment nodes and connect them to orders.
gdb.run("""
    CREATE CONSTRAINT payment_id IF NOT EXISTS
    FOR (n:Payment) REQUIRE n.transactionId IS UNIQUE
""")

payments = prep(pd.read_csv(DATA / "payment_transactions.csv"),
                dates=["transaction_date"])
load("""
UNWIND $rows AS row
MERGE (t:Payment {transactionId: row.transaction_id})
SET t.method        = row.payment_method,
    t.provider      = row.payment_provider,
    t.amount        = toFloat(row.amount),
    t.currency      = row.currency,
    t.status        = row.status,
    t.riskScore     = toFloat(row.risk_score),
    t.ipCountry     = row.ip_country,
    t.is3dSecure    = row.is_3d_secure,
    t.chargeback    = row.chargeback_raised,
    t.processingFee = toFloat(row.processing_fee),
    t.txDate        = datetime(row.transaction_date)
""", payments, label="Payment")

pay_links = prep(pd.read_csv(DATA / "payment_transactions.csv")[["transaction_id", "order_id"]])
load("""
UNWIND $rows AS row
MATCH (o:Order   {orderId:       row.order_id})
MATCH (t:Payment {transactionId: row.transaction_id})
MERGE (o)-[:PAID_BY]->(t)
""", pay_links, label="PAID_BY")

gdb.run("""
    MATCH (t:Payment)
    RETURN count(*) AS payments,
           sum(CASE WHEN t.chargeback THEN 1 ELSE 0 END) AS chargebacks,
           sum(CASE WHEN t.riskScore > 70 THEN 1 ELSE 0 END) AS highRisk,
           count(DISTINCT t.ipCountry) AS distinctIpCountries
""")

> **Notice how easy that was.** We added a new node label and a new relationship type to a live
> graph with no migration, no `ALTER TABLE`, no downtime, and no impact on any existing query.
> This is one of the most underrated practical advantages of graph databases: **schema evolution
> is additive**. Relational schema changes on a large table can lock it for hours.

## 8.2 Degree centrality — who matters most?

**Degree centrality** is simply the number of relationships a node has. Crude, but often the
single most informative number in a graph.

In [ ]:
# Which customers are most "central" in the purchase network?
gdb.run("""
    MATCH (c:Customer)
    WITH c,
         count { (c)-[:PLACED]->(:Order) }                        AS orders,
         count { (c)-[:PLACED]->(:Order)-[:CONTAINS]->(:Product) } AS lineItems,
         count { (c)-[:WROTE]->(:Review) }                        AS reviews
    WITH c, orders, lineItems, reviews, orders + lineItems + reviews AS degree
    RETURN c.name    AS customer,
           c.segment AS segment,
           orders, lineItems, reviews, degree
    ORDER BY degree DESC
    LIMIT 10
""", show_time=True)

In [ ]:
# Distribution of product degree - is it a power law?
import matplotlib.pyplot as plt

deg = gdb.run("""
    MATCH (p:Product)
    RETURN p.name AS product,
           count { (p)<-[:CONTAINS]-(:Order) } AS timesOrdered
""")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.hist(deg["timesOrdered"], bins=30, color="#F2A65A", edgecolor="#333")
ax1.set_xlabel("Times ordered"); ax1.set_ylabel("Number of products")
ax1.set_title("Product degree distribution")
ranked = deg.sort_values("timesOrdered", ascending=False).reset_index(drop=True)
ax2.plot(range(1, len(ranked) + 1), ranked["timesOrdered"], color="#4C9BE8")
ax2.set_xlabel("Product rank"); ax2.set_ylabel("Times ordered")
ax2.set_title("Rank-frequency curve")
ax2.set_xscale("log")
for a in (ax1, ax2):
    a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(deg["timesOrdered"].describe())

## 8.3 A recommendation engine, three ways

### Approach 1 — Collaborative filtering ("people like you")

The logic: find customers who bought what you bought, then recommend what *they* bought that
you haven't.

In [ ]:
def collaborative_recommendations(customer_id, limit=10):
    return gdb.run("""
        // 1. What has this customer already bought?
        MATCH (me:Customer {customerId: $cid})-[:PLACED]->(:Order)-[:CONTAINS]->(mine:Product)
        WITH me, collect(DISTINCT mine) AS myProducts

        // 2. Find peers who bought at least one of the same products
        UNWIND myProducts AS mine
        MATCH (mine)<-[:CONTAINS]-(:Order)<-[:PLACED]-(peer:Customer)
        WHERE peer <> me
        WITH me, myProducts, peer, count(DISTINCT mine) AS overlap
        WHERE overlap >= 2                       // require real similarity
        ORDER BY overlap DESC
        LIMIT 50                                 // keep only the nearest peers

        // 3. What did those peers buy that I haven't?
        MATCH (peer)-[:PLACED]->(:Order)-[:CONTAINS]->(rec:Product)
        WHERE NOT rec IN myProducts
        RETURN rec.name                     AS recommendation,
               rec.brand                    AS brand,
               rec.price                    AS price,
               rec.avgRating                AS rating,
               count(DISTINCT peer)         AS peerSupport,
               sum(overlap)                 AS weightedScore
        ORDER BY weightedScore DESC, peerSupport DESC
        LIMIT $limit
    """, {"cid": customer_id, "limit": limit}, show_time=True)


collaborative_recommendations(CID)

**Why the `overlap >= 2` filter matters.** Without it, anyone who bought a single popular item
counts as a "peer", and your recommendations collapse into a bestseller list. Requiring two or
more shared products means the peers genuinely resemble the target customer.

**Why `LIMIT 50` after ordering by overlap.** This is a *k-nearest-neighbours* cut-off. It keeps
the query fast and, more importantly, keeps the recommendations sharp — distant peers add noise.

### Approach 2 — Content-based ("more like this")

No peer behaviour at all. Recommend products that are *structurally similar* to what the
customer already likes: same category, same brand, same tags.

In [ ]:
def content_recommendations(customer_id, limit=10):
    return gdb.run("""
        MATCH (me:Customer {customerId: $cid})-[:PLACED]->(:Order)-[:CONTAINS]->(mine:Product)
        WITH me,
             collect(DISTINCT mine)         AS myProducts,
             collect(DISTINCT mine.brand)   AS myBrands

        MATCH (mine2:Product)-[:IN_CATEGORY]->(cat:Category)
        WHERE mine2 IN myProducts
        WITH me, myProducts, myBrands, collect(DISTINCT cat) AS myCategories

        MATCH (rec:Product)-[:IN_CATEGORY]->(cat2:Category)
        WHERE cat2 IN myCategories
          AND NOT rec IN myProducts
          AND rec.isActive = true
        RETURN rec.name    AS recommendation,
               rec.brand   AS brand,
               cat2.name   AS category,
               rec.price   AS price,
               rec.avgRating AS rating,
               // score: brand affinity + rating + a nudge for featured items
               round(
                 (CASE WHEN rec.brand IN myBrands THEN 3.0 ELSE 0.0 END)
                 + coalesce(rec.avgRating, 0)
                 + (CASE WHEN rec.isFeatured THEN 0.5 ELSE 0.0 END)
               , 2) AS score
        ORDER BY score DESC, rating DESC
        LIMIT $limit
    """, {"cid": customer_id, "limit": limit}, show_time=True)


content_recommendations(CID)

### Approach 3 — Jaccard similarity (a proper similarity metric)

The **Jaccard index** measures how much two sets overlap:

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

For customers, `A` and `B` are the sets of products each bought. Jaccard is better than raw
overlap because it **normalises for activity level** — a customer who bought 200 products
shouldn't look similar to everyone just because they bought a lot.

In [ ]:
# Jaccard similarity between customers, in pure Cypher
similar = gdb.run("""
    MATCH (me:Customer {customerId: $cid})-[:PLACED]->(:Order)-[:CONTAINS]->(p:Product)
    WITH me, collect(DISTINCT p) AS setA

    MATCH (other:Customer)-[:PLACED]->(:Order)-[:CONTAINS]->(q:Product)
    WHERE other <> me
    WITH me, setA, other, collect(DISTINCT q) AS setB

    // intersection and union, computed with list comprehensions
    WITH me, other, setA, setB,
         size([x IN setA WHERE x IN setB]) AS intersection
    WITH other, intersection,
         size(setA) + size(setB) - intersection AS unionSize,
         size(setA) AS sizeA, size(setB) AS sizeB
    WHERE intersection > 0
    RETURN other.customerId                              AS customerId,
           other.segment                                 AS segment,
           sizeA, sizeB, intersection, unionSize,
           round(toFloat(intersection) / unionSize, 4)   AS jaccard
    ORDER BY jaccard DESC
    LIMIT 10
""", {"cid": CID}, show_time=True)
similar

> **Performance warning, and it's an important one.** That query compares one customer against
> *every* other customer — it's `O(n)` in customers, and a full pairwise similarity matrix would
> be `O(n²)`. At 1,201 customers that's fine. At a million customers it is not.
>
> Production systems solve this by **pre-computing** similarity and storing it as a relationship:
> `(:Customer)-[:SIMILAR_TO {score: 0.34}]->(:Customer)`, refreshed nightly. Then recommendation
> at query time is a single hop. We'll build exactly that in Part 10.

In [ ]:
# Compare all three recommenders side by side for the same customer
collab = collaborative_recommendations(CID, 10)[["recommendation", "weightedScore"]]
content = content_recommendations(CID, 10)[["recommendation", "score"]]

comparison = pd.DataFrame({
    "Collaborative": collab["recommendation"].tolist(),
    "Content-based": content["recommendation"].tolist(),
})
print("Two recommenders, same customer - notice how little they agree:\n")
display(comparison)

overlap = set(collab["recommendation"]) & set(content["recommendation"])
print("\nProducts recommended by BOTH: {}".format(overlap if overlap else "none"))
print("\nThis disagreement is normal and useful. Real systems blend several")
print("signals - a 'hybrid recommender' - precisely because each one is blind")
print("in a different way.")

## 8.4 Anomaly and fraud pattern detection

Fraud detection is the flagship graph use case, because fraud is fundamentally about
**connections**: shared devices, shared addresses, shared cards, circular money flows.

Let's find genuine anomalies in our data.

In [ ]:
# Signal 1: chargebacks - the clearest fraud indicator we have
gdb.run("""
    MATCH (c:Customer)-[:PLACED]->(o:Order)-[:PAID_BY]->(t:Payment)
    WHERE t.chargeback = true
    RETURN c.name         AS customer,
           c.segment      AS segment,
           c.city         AS city,
           o.orderId      AS orderId,
           round(t.amount, 2) AS amount,
           t.method       AS paymentMethod,
           t.riskScore    AS riskScore,
           t.ipCountry    AS ipCountry,
           t.is3dSecure   AS threeDSecure,
           o.status       AS orderStatus
    ORDER BY amount DESC
""")

In [ ]:
# Signal 2: a composite risk score built from several weak signals.
# No single signal proves fraud - but several together are worth investigating.
gdb.run("""
    MATCH (c:Customer)-[:PLACED]->(o:Order)-[:PAID_BY]->(t:Payment)
    WITH c,
         count(o)                                                    AS orders,
         sum(CASE WHEN t.chargeback THEN 1 ELSE 0 END)               AS chargebacks,
         sum(CASE WHEN t.riskScore > 70 THEN 1 ELSE 0 END)           AS highRiskPayments,
         sum(CASE WHEN t.ipCountry <> 'US' THEN 1 ELSE 0 END)        AS foreignIpOrders,
         sum(CASE WHEN o.status = 'Returned' THEN 1 ELSE 0 END)      AS returns,
         sum(CASE WHEN NOT t.is3dSecure THEN 1 ELSE 0 END)           AS no3ds,
         sum(o.totalAmount)                                          AS totalValue
    WITH c, orders, chargebacks, highRiskPayments, foreignIpOrders, returns, totalValue,
         (chargebacks * 40)
       + (highRiskPayments * 15)
       + (foreignIpOrders * 3)
       + (returns * 8)                                               AS riskScore
    WHERE riskScore >= 30
    RETURN c.name        AS customer,
           c.segment     AS segment,
           orders, chargebacks, highRiskPayments, foreignIpOrders, returns,
           round(totalValue, 2) AS totalValue,
           riskScore
    ORDER BY riskScore DESC
    LIMIT 15
""", show_time=True)

> **Be honest about what this is.** That "risk score" uses hand-picked weights. It is a
> *heuristic*, not a model. In production you'd use these graph features as **inputs to a trained
> model**, and the graph's job is to compute features that are expensive or impossible in SQL —
> "number of accounts within 3 hops that had a chargeback", for example.

In [ ]:
# Signal 3: RETURN ABUSE - customers whose returns are concentrated on expensive items.
# This is a classic retail fraud pattern (wardrobing / receipt fraud).
gdb.run("""
    MATCH (c:Customer)-[:PLACED]->(o:Order)
    WITH c,
         count(o)                                               AS totalOrders,
         sum(CASE WHEN o.status = 'Returned' THEN 1 ELSE 0 END) AS returns,
         sum(CASE WHEN o.status = 'Returned' THEN o.totalAmount ELSE 0 END) AS returnedValue,
         sum(o.totalAmount)                                     AS totalValue
    WHERE totalOrders >= 4 AND returns >= 2
    WITH c, totalOrders, returns, returnedValue, totalValue,
         round(toFloat(returns) / totalOrders, 3)     AS returnRate,
         round(returnedValue / totalValue, 3)         AS valueReturnRate
    WHERE returnRate >= 0.4
    RETURN c.name    AS customer,
           c.segment AS segment,
           totalOrders, returns, returnRate,
           round(returnedValue, 2) AS returnedValue,
           valueReturnRate
    ORDER BY valueReturnRate DESC, returnRate DESC
    LIMIT 15
""")

### The pattern that isn't in this dataset — and why that's worth knowing

The most valuable fraud query of all is **ring detection**: several accounts that look
independent but share a physical address, device fingerprint, or payment card.

**Our dataset was synthetically generated with unique addresses per customer, so no rings exist
in it.** Rather than pretend otherwise, let's *create* one — which also demonstrates how you'd
model shared identifiers in the first place.

In [ ]:
# Promote shared identifiers to NODES, so that "sharing" becomes a traversable path.
# We seed a small synthetic fraud ring to demonstrate the detection pattern.

gdb.run("""
    CREATE CONSTRAINT device_id IF NOT EXISTS
    FOR (n:Device) REQUIRE n.fingerprint IS UNIQUE
""")

# Give five real customers a shared device fingerprint (the ring),
# and give 200 others their own unique device (the honest majority).
gdb.run("""
    MATCH (c:Customer)
    WITH c ORDER BY c.customerId LIMIT 5
    MERGE (d:Device {fingerprint: 'FP-SHARED-9x8c2'})
    MERGE (c)-[:USED_DEVICE]->(d)
""", show_time=True)

gdb.run("""
    MATCH (c:Customer)
    WITH c ORDER BY c.customerId SKIP 5 LIMIT 200
    MERGE (d:Device {fingerprint: 'FP-' + toString(c.customerId)})
    MERGE (c)-[:USED_DEVICE]->(d)
""", show_time=True)

print("Seeded a device graph: 1 shared fingerprint + 200 unique ones.")

In [ ]:
# NOW the ring detection query - and notice how short it is.
ring = gdb.run("""
    MATCH (d:Device)<-[:USED_DEVICE]-(c:Customer)
    WITH d, collect(c) AS members, count(c) AS memberCount
    WHERE memberCount > 1
    UNWIND members AS m
    RETURN d.fingerprint    AS sharedDevice,
           memberCount,
           m.name           AS customer,
           m.city           AS city,
           m.segment        AS segment,
           count { (m)-[:PLACED]->(:Order) } AS orders
    ORDER BY memberCount DESC, customer
""")
display(ring)

draw("""
    MATCH path = (d:Device {fingerprint: 'FP-SHARED-9x8c2'})<-[:USED_DEVICE]-(:Customer)
    RETURN path
""", title="A fraud ring: five accounts sharing one device", figsize=(10, 6))

### Why this is the argument for graph databases

Read that ring query again. **Four lines.** And critically — its cost depends on the number of
*devices*, not the number of *customers*.

The relational equivalent is a self-join of the customers table against itself on
`device_fingerprint`, which is `O(n²)` in the worst case and needs careful indexing to be
tolerable. And when the fraud team asks the follow-up question — *"now find accounts that share a
device with an account that shares an address with a known fraudster"* — the SQL becomes
effectively unwritable, while the Cypher just grows another hop:

```cypher
MATCH (known:Customer {flagged: true})-[:USED_DEVICE|LIVES_AT*1..4]-(suspect:Customer)
RETURN DISTINCT suspect
```

> **This is why every major bank, payment processor and insurer now runs a graph database
> somewhere in their fraud stack.** Not because it's faster at any one query, but because it
> makes the *next* question askable.

## 8.5 Exercises — Part 8

**Exercise 8.1** — Build a **product co-purchase** network: pairs of products bought by the same
customer. Return the top 15 pairs by the number of shared customers.

<details>
<summary><b>&#128161; Show solution &mdash; 8.1</b></summary>

```cypher
MATCH (c:Customer)-[:PLACED]->(:Order)-[:CONTAINS]->(p1:Product)
MATCH (c)-[:PLACED]->(:Order)-[:CONTAINS]->(p2:Product)
WHERE p1.productId < p2.productId          // avoid duplicates and self-pairs
WITH p1, p2, count(DISTINCT c) AS sharedCustomers
WHERE sharedCustomers >= 3
RETURN p1.name AS productA, p2.name AS productB,
       p1.brand AS brandA, p2.brand AS brandB,
       sharedCustomers
ORDER BY sharedCustomers DESC
LIMIT 15
```
**The `p1.productId < p2.productId` trick is essential.** Without it you get every pair twice
(A-B and B-A) plus every product paired with itself. Using `<` rather than `<>` gives you each
unordered pair exactly once — memorise this idiom, you'll need it in every similarity query.

</details>

**Exercise 8.2** — Find "gateway products": products most often bought by customers on their
**first** order. These are the products that acquire customers.

<details>
<summary><b>&#128161; Show solution &mdash; 8.2</b></summary>

```cypher
MATCH (c:Customer)-[:PLACED]->(o:Order)
WITH c, min(o.orderDate) AS firstOrderDate
MATCH (c)-[:PLACED]->(first:Order)-[:CONTAINS]->(p:Product)
WHERE first.orderDate = firstOrderDate
WITH p, count(DISTINCT c) AS acquiredCustomers
RETURN p.name  AS gatewayProduct,
       p.brand AS brand,
       p.price AS price,
       acquiredCustomers,
       count { (p)<-[:CONTAINS]-(:Order) } AS totalOrders,
       round(toFloat(acquiredCustomers) /
             count { (p)<-[:CONTAINS]-(:Order) }, 3) AS acquisitionRatio
ORDER BY acquiredCustomers DESC
LIMIT 15
```
The `acquisitionRatio` is the interesting column: a high ratio means the product is
*disproportionately* a first purchase, which is far more actionable for marketing than raw volume.

</details>

**Exercise 8.3** *(harder)* — Compute **category affinity**: for each pair of top-level
categories, what fraction of customers who bought from category A also bought from category B?
This is the "customers who shop Electronics also shop..." analysis.

<details>
<summary><b>&#128161; Show solution &mdash; 8.3</b></summary>

```cypher
// Map every customer to the set of top-level categories they shopped
MATCH (c:Customer)-[:PLACED]->(:Order)-[:CONTAINS]->(:Product)
      -[:IN_CATEGORY]->(leaf:Category)
MATCH (root:Category)-[:PARENT_OF*0..2]->(leaf)
WHERE root.level = 1
WITH c, collect(DISTINCT root.name) AS cats

// Count customers per category
UNWIND cats AS catA
WITH c, cats, catA
UNWIND cats AS catB
WITH catA, catB, count(DISTINCT c) AS both
WHERE catA <> catB

// Total customers per category, for the denominator
MATCH (cust:Customer)-[:PLACED]->(:Order)-[:CONTAINS]->(:Product)
      -[:IN_CATEGORY]->(l:Category)
MATCH (r:Category)-[:PARENT_OF*0..2]->(l)
WHERE r.level = 1 AND r.name = catA
WITH catA, catB, both, count(DISTINCT cust) AS totalA
RETURN catA                                  AS ifTheyBought,
       catB                                  AS theyAlsoBought,
       both                                  AS customers,
       totalA,
       round(toFloat(both) / totalA, 3)      AS affinity
ORDER BY affinity DESC
LIMIT 20
```
This exercises the **`collect` then double-`UNWIND`** pattern for generating pairs from a set —
the same technique behind market-basket analysis and association rules.

</details>

## 8.6 Checkpoint 8

In [ ]:
cb = gdb.run("MATCH (t:Payment) WHERE t.chargeback RETURN count(t) AS n")
rec = collaborative_recommendations(CID, 5)
jac = gdb.run("""
    MATCH (me:Customer {customerId:$cid})-[:PLACED]->(:Order)-[:CONTAINS]->(p:Product)
    WITH me, collect(DISTINCT p) AS a
    MATCH (o:Customer)-[:PLACED]->(:Order)-[:CONTAINS]->(q:Product)
    WHERE o <> me
    WITH a, o, collect(DISTINCT q) AS b
    WITH o, size([x IN a WHERE x IN b]) AS i, size(a)+size(b) AS tot
    WHERE i > 0 RETURN count(o) AS n
""", {"cid": CID})
rings = gdb.run("""
    MATCH (d:Device)<-[:USED_DEVICE]-(c:Customer)
    WITH d, count(c) AS n WHERE n > 1 RETURN count(d) AS rings
""")

checkpoint("Part 8 - Graph analytics", [
    ("Payment nodes loaded",              int(cb.iloc[0]["n"]) > 0),
    ("Collaborative recommender works",   len(rec) > 0),
    ("Jaccard finds similar customers",   int(jac.iloc[0]["n"]) > 0),
    ("Fraud ring detected",               int(rings.iloc[0]["rings"]) == 1),
])

---
# Part 9 — Performance and Query Tuning

Everything has been fast so far because our graph is small. Production graphs are not. This part
teaches you to *see* what the database is doing rather than guess.

## 9.1 `EXPLAIN` and `PROFILE`

| Command | Runs the query? | Gives real row counts? | Use when |
|---|---|---|---|
| `EXPLAIN` | No | No — estimates only | Checking the plan of a slow/dangerous query |
| `PROFILE` | **Yes** | **Yes** — actual rows and db hits | Diagnosing a query you can afford to run |

The number that matters most is **db hits**: how many times the engine touched the storage layer.
Fewer is better. A query with 10 million db hits returning 5 rows is doing something wrong.

In [ ]:
# A helper to print query plans, since to_df() only returns result rows.
def show_plan(cypher, params=None, profile=True):
    """Print the execution plan for a query as an indented tree."""
    keyword = "PROFILE" if profile else "EXPLAIN"
    with gdb.driver.session(database=gdb.database) as session:
        result = session.run(keyword + "\n" + cypher, params or {})
        list(result)                       # consume so PROFILE gets real counters
        summary = result.consume()
    plan = summary.profile or summary.plan

    def walk(node, depth=0):
        args = node.get("args", {})
        rows = node.get("rows", args.get("EstimatedRows"))
        hits = node.get("dbHits")
        detail = args.get("Details", "")
        if isinstance(detail, str) and len(detail) > 62:
            detail = detail[:60] + ".."
        bits = []
        if rows is not None:
            bits.append("rows={}".format(int(rows)))
        if hits is not None:
            bits.append("dbHits={}".format(int(hits)))
        print("{}{:<26s} {:<28s} {}".format(
            "  " * depth, node["operatorType"], detail, "  ".join(bits)))
        for child in node.get("children", []):
            walk(child, depth + 1)

    walk(plan)
    if summary.profile:
        print("\nTotal db hits: {:,}".format(
            sum_hits(plan)))


def sum_hits(node):
    return (node.get("dbHits") or 0) + sum(sum_hits(c) for c in node.get("children", []))


print("Plan helper ready.")

In [ ]:
# GOOD PLAN: filtering on an indexed property.
print("=== Indexed lookup (customerId has a uniqueness constraint) ===\n")
show_plan("MATCH (c:Customer {customerId: $cid}) RETURN c.name", {"cid": CID})

In [ ]:
# BAD PLAN: filtering on a property with no index.
print("=== Unindexed lookup (email has no index) ===\n")
show_plan("MATCH (c:Customer {email: 'nobody@example.com'}) RETURN c.name")

### Reading those two plans

Look for the **leaf operator** — the bottom of the tree, where the query starts:

| Operator | Meaning | Verdict |
|---|---|---|
| `NodeUniqueIndexSeek` | Jumped straight to one node via a unique index | Best |
| `NodeIndexSeek` | Used an index to find a small set | Good |
| `NodeIndexScan` | Read the whole index | Acceptable |
| `NodeByLabelScan` | Read **every** node with that label | Bad at scale |
| `AllNodesScan` | Read **every node in the database** | Always a bug |

The unindexed query above uses `NodeByLabelScan` and touches every `Customer`. At 1,201 nodes
that's imperceptible. At 50 million it's a production incident.

In [ ]:
# Fix it: add an index and watch the plan change.
gdb.run("CREATE INDEX customer_email IF NOT EXISTS FOR (n:Customer) ON (n.email)")
import time; time.sleep(2)                   # index population is asynchronous

print("=== Same query, now with an index on email ===\n")
show_plan("MATCH (c:Customer {email: 'nobody@example.com'}) RETURN c.name")

## 9.2 The index types

| Type | Syntax | Use for |
|---|---|---|
| **Range** (default) | `CREATE INDEX x FOR (n:L) ON (n.p)` | Equality, ranges, `STARTS WITH`, ordering |
| **Composite** | `... ON (n.a, n.b)` | Queries filtering both properties together |
| **Text** | `CREATE TEXT INDEX ...` | `CONTAINS` and `ENDS WITH` |
| **Point** | `CREATE POINT INDEX ...` | Spatial proximity and bounding-box queries |
| **Full-text** | `CREATE FULLTEXT INDEX ...` | Natural-language search across properties |
| **Relationship** | `FOR ()-[r:TYPE]-() ON (r.p)` | Filtering on relationship properties |

All of these work on Aura Free.

In [ ]:
# A full-text index over review text - this is real search, with scoring.
gdb.run("""
    CREATE FULLTEXT INDEX reviewSearch IF NOT EXISTS
    FOR (n:Review) ON EACH [n.title, n.text]
""")
time.sleep(2)

gdb.run("""
    CALL db.index.fulltext.queryNodes('reviewSearch', 'comfortable AND stitching')
    YIELD node, score
    MATCH (node)-[:REVIEWS]->(p:Product)
    RETURN round(score, 3) AS relevance,
           node.rating     AS rating,
           node.title      AS title,
           p.name          AS product
    ORDER BY relevance DESC
    LIMIT 10
""", show_time=True)

In [ ]:
# A relationship-property index
gdb.run("""
    CREATE INDEX contains_quantity IF NOT EXISTS
    FOR ()-[r:CONTAINS]-() ON (r.quantity)
""")
time.sleep(2)

show_plan("""
    MATCH ()-[r:CONTAINS]->(p:Product)
    WHERE r.quantity >= 5
    RETURN p.name, r.quantity
    LIMIT 10
""")

## 9.3 The anti-patterns

In [ ]:
import time

# ANTI-PATTERN 1: expanding first and filtering last.
slow = """
    MATCH (c:Customer)-[:PLACED]->(o:Order)-[:CONTAINS]->(p:Product)
    WHERE c.segment = 'VIP' AND p.price > 1000
    RETURN count(*) AS n
"""
# BETTER: filter as early as possible, and cut the row count with WITH.
fast = """
    MATCH (c:Customer {segment: 'VIP'})
    MATCH (c)-[:PLACED]->(o:Order)
    MATCH (o)-[:CONTAINS]->(p:Product)
    WHERE p.price > 1000
    RETURN count(*) AS n
"""
for name, q in [("expand-then-filter", slow), ("filter-early", fast)]:
    t0 = time.perf_counter()
    r = gdb.run(q)
    print("{:<22s} {:7.1f} ms   result={}".format(
        name, (time.perf_counter() - t0) * 1000, int(r.iloc[0]["n"])))

print("\nOn a graph this small the difference is noise. The PLANS differ, though -")
print("compare them below, and imagine 10 million customers.")

In [ ]:
print("=== Plan: expand-then-filter ===\n")
show_plan(slow)
print("\n\n=== Plan: filter-early ===\n")
show_plan(fast)

### The anti-pattern checklist

| Anti-pattern | Why it hurts | Fix |
|---|---|---|
| Unbounded `*` in a variable-length path | Explores an exponential frontier | Always bound: `*1..4` |
| No label in a pattern: `MATCH (n)` | Forces `AllNodesScan` | Always label your nodes |
| String-concatenating parameters | Defeats the plan cache; injection risk | Use `$param` |
| `MATCH` everything then `WHERE` | Builds huge intermediate results | Filter in the pattern, or filter early |
| Missing `WITH ... LIMIT` before an expensive expansion | Expands rows you'll discard | Narrow before you traverse |
| `count(x)` where `count(DISTINCT x)` was meant | Silently wrong numbers | Ask "what does one row represent?" |
| Creating indexes *after* bulk loading | The load itself was slow | Constraints and indexes first |
| Storing lists that only grow | Property updates rewrite the whole list | Model list members as nodes |

## 9.4 Cardinality: the mental model that makes you fast

The single most useful habit is to **track how many rows exist after each clause**.

```cypher
MATCH (c:Customer)                        // 1,201 rows
MATCH (c)-[:PLACED]->(o:Order)            // 6,030 rows   (fan-out ×5)
MATCH (o)-[:CONTAINS]->(p:Product)        // 14,456 rows  (fan-out ×2.4)
MATCH (p)<-[:SUPPLIES]-(s:Supplier)       // ~24,000 rows (fan-out ×1.7)
```

Each `MATCH` **multiplies**. Four hops turned 1,201 rows into 24,000. Add two more hops and
you're in the millions — on a graph with only 12,000 nodes.

**The fix is always the same: aggregate or `LIMIT` at the earliest point where you can.**

In [ ]:
# Watch the cardinality explode, stage by stage.
stages = [
    ("MATCH (c:Customer)", "MATCH (c:Customer) RETURN count(*) AS n"),
    ("+ PLACED->Order",
     "MATCH (c:Customer)-[:PLACED]->(o:Order) RETURN count(*) AS n"),
    ("+ CONTAINS->Product",
     "MATCH (c:Customer)-[:PLACED]->(:Order)-[:CONTAINS]->(p:Product) "
     "RETURN count(*) AS n"),
    ("+ <-SUPPLIES-Supplier",
     "MATCH (c:Customer)-[:PLACED]->(:Order)-[:CONTAINS]->(p:Product)"
     "<-[:SUPPLIES]-(s:Supplier) RETURN count(*) AS n"),
]
prev = None
for label, q in stages:
    t0 = time.perf_counter()
    n = int(gdb.run(q).iloc[0]["n"])
    ms = (time.perf_counter() - t0) * 1000
    factor = "" if prev is None else "   (x{:.1f})".format(n / prev)
    print("{:<26s} {:>9,} rows   {:6.1f} ms{}".format(label, n, ms, factor))
    prev = n

## 9.5 Exercises — Part 9

**Exercise 9.1** — `PROFILE` this query, then rewrite it to use fewer db hits:

```cypher
MATCH (c:Customer)-[:PLACED]->(o:Order)-[:CONTAINS]->(p:Product)
WHERE c.city = 'Boston' AND p.brand = 'Nike' AND o.status = 'Delivered'
RETURN c.name, p.name
```

<details>
<summary><b>&#128161; Show solution &mdash; 9.1</b></summary>

```cypher
// Original: PROFILE it first to get a baseline.
// Improvements:
//   1. c.city is indexed (we created customer_city in Part 3) - anchor there
//   2. Filter each node in the pattern rather than all at the end
//   3. p.brand is indexed too, so let the planner choose the cheaper anchor
PROFILE
MATCH (c:Customer {city: 'Boston'})
MATCH (c)-[:PLACED]->(o:Order {status: 'Delivered'})
MATCH (o)-[:CONTAINS]->(p:Product {brand: 'Nike'})
RETURN c.name AS customer, p.name AS product
```
Look for the leaf operator to change from `NodeByLabelScan` to `NodeIndexSeek`, and total
db hits to drop. If `Boston` has few customers, anchoring there is the winning plan; if it has
many, anchoring on the brand may be better. **The planner usually gets this right — your job is
to make sure the indexes exist so it has good options.**

</details>

**Exercise 9.2** — This query is dangerous. Explain why, then fix it.

```cypher
MATCH (c:Customer)-[*]-(other:Customer)
RETURN DISTINCT other.name
```

<details>
<summary><b>&#128161; Show solution &mdash; 9.2</b></summary>

**Three separate problems:**

1. **`[*]` is unbounded.** It explores paths of any length, following *every* relationship type
   in both directions. On a connected graph this can visit essentially every path in the
   database — combinatorial blow-up.
2. **No relationship types.** It traverses `PLACED`, `CONTAINS`, `REVIEWS`, `SUPPLIES`,
   `STOCKS`, `SHIPPED_FROM`, `PAID_BY` — including through high-degree hub nodes like
   warehouses, which connect to almost everything and destroy any hope of pruning.
3. **No anchor.** It starts from *every* customer.

**Fixed:**
```cypher
MATCH (c:Customer {customerId: $cid})
MATCH (c)-[:PLACED|CONTAINS*1..4]-(other:Customer)
WHERE other <> c
RETURN DISTINCT other.name
LIMIT 100
```
Bound the depth, name the relationship types, anchor on one node, and cap the output.
All four fixes matter.

</details>

**Exercise 9.3** — Create a composite index that would help this query, then verify with
`EXPLAIN` that the planner uses it:

```cypher
MATCH (p:Product)
WHERE p.brand = 'Nike' AND p.price > 500
RETURN p.name
```

<details>
<summary><b>&#128161; Show solution &mdash; 9.3</b></summary>

```cypher
CREATE INDEX product_brand_price IF NOT EXISTS
FOR (n:Product) ON (n.brand, n.price);
```
Then:
```cypher
EXPLAIN
MATCH (p:Product)
WHERE p.brand = 'Nike' AND p.price > 500
RETURN p.name
```
You should see `NodeIndexSeekByRange` referencing the composite index.

**The rule for composite index column order:** put the property used for **equality** first, and
the property used for **ranges** second. A composite index can seek on a prefix of its
properties, so `(brand, price)` serves `brand = ...` and `brand = ... AND price > ...`, but
**not** `price > ...` on its own.

</details>

## 9.6 Checkpoint 9

In [ ]:
idx = gdb.run("SHOW INDEXES YIELD name, type, labelsOrTypes, properties RETURN *")
ft = gdb.run("""
    CALL db.index.fulltext.queryNodes('reviewSearch', 'comfortable')
    YIELD node, score RETURN count(*) AS n
""")

checkpoint("Part 9 - Performance", [
    ("At least 10 indexes exist",       len(idx) >= 10),
    ("Full-text index returns results", int(ft.iloc[0]["n"]) > 0),
    ("show_plan() works",               True),
    ("I can name 3 anti-patterns",      True),
])

display(idx)

---
# Part 10 — Writing and Refactoring the Graph

Everything so far has been read-only. Now we change the graph.

## 10.1 The write clauses

| Clause | Effect |
|---|---|
| `CREATE` | Always creates. Duplicates are possible. |
| `MERGE` | Match-or-create (upsert) on the **whole pattern** |
| `SET` | Set properties, or add labels |
| `REMOVE` | Remove properties or labels |
| `DELETE` | Delete a node or relationship — **fails** if the node still has relationships |
| `DETACH DELETE` | Delete a node **and** all its relationships |

In [ ]:
# CREATE vs MERGE, demonstrated. Run this cell twice and watch what happens.
gdb.run("CREATE (t:Demo {name: 'created'})", show_time=True)
gdb.run("MERGE  (t:Demo {name: 'merged'})",  show_time=True)

gdb.run("""
    MATCH (t:Demo)
    RETURN t.name AS name, count(*) AS copies
    ORDER BY name
""")

Run the cell above a second time. `created` becomes 2, then 3, then 4... while `merged` stays
at 1. That is the entire difference, and it's why every loader in this notebook used `MERGE`.

In [ ]:
# ON CREATE SET / ON MATCH SET - different behaviour for insert vs update.
# This is the idiomatic "upsert with audit fields" pattern.
for run in (1, 2):
    gdb.run("""
        MERGE (t:Demo {name: 'audited'})
        ON CREATE SET t.createdAt = datetime(), t.timesSeen = 1
        ON MATCH  SET t.updatedAt = datetime(), t.timesSeen = t.timesSeen + 1
    """)
    df = gdb.run("MATCH (t:Demo {name:'audited'}) "
                 "RETURN t.timesSeen AS timesSeen, "
                 "t.createdAt IS NOT NULL AS hasCreatedAt, "
                 "t.updatedAt IS NOT NULL AS hasUpdatedAt")
    print("run {}: {}".format(run, df.to_dict("records")[0]))

In [ ]:
# Clean up the demo nodes.
gdb.run("MATCH (t:Demo) DETACH DELETE t", show_time=True)
print("Demo nodes removed.")

## 10.2 Labels as mutable state

A powerful and underused technique: **labels can be added and removed at runtime**. Because
labels are indexed, this turns an expensive filter into a cheap lookup.

In [ ]:
# Tag high-value customers with an extra label.
gdb.run("""
    MATCH (c:Customer)-[:PLACED]->(o:Order)
    WITH c, sum(o.totalAmount) AS spend
    WHERE spend > 25000
    SET c:HighValue, c.computedSpend = round(spend, 2)
    RETURN count(c) AS tagged
""", show_time=True)

# Now querying them is a label scan over a small set, not a scan-plus-aggregate.
gdb.run("""
    MATCH (c:HighValue)
    RETURN c.name AS customer, c.segment AS segment,
           c.computedSpend AS spend, labels(c) AS labels
    ORDER BY spend DESC
    LIMIT 8
""", show_time=True)

> **When to use a label instead of a property:**
> - The value is **boolean-ish** (`:Churned`, `:Verified`, `:Suspended`)
> - You filter on it **constantly**
> - Only a **minority** of nodes have it (labels index best when selective)
>
> **When not to:** if it's a value with many possible states, or it changes on every write.
> Label changes are cheap but not free, and thousands of labels harm the planner.

In [ ]:
# REMOVE takes labels off again.
gdb.run("MATCH (c:HighValue) REMOVE c:HighValue RETURN count(c) AS untagged",
        show_time=True)

## 10.3 Graph refactoring 1 — promote a property to a node

Right now `brand` is a **string property** on `Product`. That's fine for filtering
(`WHERE p.brand = 'Nike'`), but it can't be traversed. You can't ask *"which brands does this
customer's peer group prefer?"* without an aggregation over every product.

Let's promote it. This is one of the most common refactorings you'll ever perform on a graph.

In [ ]:
# Step 1: create a Brand node for each distinct brand string.
gdb.run("""
    CREATE CONSTRAINT brand_name IF NOT EXISTS
    FOR (b:Brand) REQUIRE b.name IS UNIQUE
""")

gdb.run("""
    MATCH (p:Product)
    WHERE p.brand IS NOT NULL
    MERGE (b:Brand {name: p.brand})
    MERGE (p)-[:MADE_BY]->(b)
""", show_time=True)

gdb.run("""
    MATCH (b:Brand)
    RETURN count(b) AS brands,
           avg(count { (b)<-[:MADE_BY]-(:Product) }) AS avgProductsPerBrand
""")

In [ ]:
# Step 2: now brand-level questions become traversals.
gdb.run("""
    MATCH (c:Customer {customerId: $cid})-[:PLACED]->(:Order)-[:CONTAINS]->(:Product)
          -[:MADE_BY]->(b:Brand)
    WITH b, count(*) AS purchases
    ORDER BY purchases DESC LIMIT 3

    // ...and find other customers loyal to the same brands
    MATCH (b)<-[:MADE_BY]-(:Product)<-[:CONTAINS]-(:Order)<-[:PLACED]-(peer:Customer)
    WHERE peer.customerId <> $cid
    RETURN b.name AS brand, purchases AS myPurchases,
           count(DISTINCT peer) AS otherCustomers
    ORDER BY myPurchases DESC
""", {"cid": CID}, show_time=True)

> **Should you now delete `p.brand`?** Usually **no**. Keeping the property *and* the
> relationship is a deliberate, common trade-off: the property makes simple filters fast, the
> relationship makes traversals possible. The cost is that you must keep them in sync on write.
> **Denormalisation is as legitimate in graphs as it is anywhere else** — just be explicit about it.

## 10.4 Graph refactoring 2 — pre-compute an expensive relationship

In Part 8 we noted that pairwise similarity is `O(n²)` and shouldn't be computed at query time.
The standard solution: compute it in a batch job and **store the result as a relationship**.

In [ ]:
# Materialise SIMILAR_TO relationships between customers who share >= 3 products.
# In production this would be a nightly job; here it runs in seconds.
gdb.run("MATCH ()-[r:SIMILAR_TO]->() DELETE r")     # idempotent re-run

gdb.run("""
    MATCH (a:Customer)-[:PLACED]->(:Order)-[:CONTAINS]->(p:Product)
    WITH a, collect(DISTINCT p) AS setA
    WHERE size(setA) >= 3

    MATCH (b:Customer)-[:PLACED]->(:Order)-[:CONTAINS]->(q:Product)
    WHERE b.customerId > a.customerId          // each pair once
    WITH a, setA, b, collect(DISTINCT q) AS setB
    WHERE size(setB) >= 3

    WITH a, b,
         size([x IN setA WHERE x IN setB])                    AS shared,
         size(setA) + size(setB)                              AS combined
    WHERE shared >= 3
    WITH a, b, shared,
         toFloat(shared) / (combined - shared)                AS jaccard

    MERGE (a)-[s:SIMILAR_TO]->(b)
    SET s.jaccard = round(jaccard, 4), s.sharedProducts = shared,
        s.computedAt = datetime()
    RETURN count(s) AS similarityEdgesCreated
""", show_time=True)

In [ ]:
# Recommendation is now ONE HOP. Compare the timing to the Part 8 version.
gdb.run("""
    MATCH (me:Customer {customerId: $cid})-[s:SIMILAR_TO]-(peer:Customer)
    MATCH (peer)-[:PLACED]->(:Order)-[:CONTAINS]->(rec:Product)
    WHERE NOT (me)-[:PLACED]->(:Order)-[:CONTAINS]->(rec)
    RETURN rec.name              AS recommendation,
           rec.brand             AS brand,
           round(sum(s.jaccard), 4) AS score,
           count(DISTINCT peer)  AS peers
    ORDER BY score DESC
    LIMIT 10
""", {"cid": CID}, show_time=True)

In [ ]:
draw("""
    MATCH path = (a:Customer)-[:SIMILAR_TO]-(b:Customer)
    WITH path LIMIT 30
    RETURN path
""", title="The materialised customer similarity network", figsize=(13, 9))

> **This is the single most important production pattern in this notebook.** Expensive graph
> computations get **materialised as relationships** during off-peak batch runs, and the online
> query becomes a one-hop lookup. Recommendation engines, fraud scores, and knowledge-graph
> inference all work this way.
>
> The trade-off is **staleness**: the similarity edges reflect the data as of `computedAt`.
> That's why we stored a timestamp on the relationship — always record when a derived value
> was derived.

## 10.5 Deleting safely

In [ ]:
# DELETE refuses to orphan relationships - a genuinely useful safety net.
try:
    gdb.run("MATCH (c:Customer {customerId: $cid}) DELETE c", {"cid": CID})
except Exception as exc:
    print("As expected, plain DELETE was refused:\n")
    print(str(exc)[:300])

print("\n\nUse DETACH DELETE when you really mean it - but note that it")
print("silently removes every attached relationship too.")

In [ ]:
# Safe pattern for bulk deletes: batch them so the transaction stays small.
# (We delete nothing real here - this demonstrates the shape.)
demo = """
    MATCH (n:SomeLabelThatDoesNotExist)
    WITH n LIMIT 5000
    DETACH DELETE n
    RETURN count(n) AS deleted
"""
print("Batched delete template:\n" + demo)
print("Loop it in Python until it returns 0 - exactly as our Part 3 clear-down did.")

## 10.6 Exercises — Part 10

**Exercise 10.1** — Promote the product `tags` list into `(:Tag)` nodes with a
`(:Product)-[:TAGGED]->(:Tag)` relationship. Then find the tags most associated with
high-rated products.

<details>
<summary><b>&#128161; Show solution &mdash; 10.1</b></summary>

```cypher
// Create the Tag nodes
CREATE CONSTRAINT tag_name IF NOT EXISTS FOR (t:Tag) REQUIRE t.name IS UNIQUE;

MATCH (p:Product)
UNWIND p.tags AS tagName
WITH p, trim(tagName) AS tagName
WHERE tagName <> ''
MERGE (t:Tag {name: tagName})
MERGE (p)-[:TAGGED]->(t);

// Which tags correlate with high ratings?
MATCH (t:Tag)<-[:TAGGED]-(p:Product)
WITH t, count(p) AS products, avg(p.avgRating) AS avgRating
WHERE products >= 10
RETURN t.name AS tag, products, round(avgRating, 3) AS avgRating
ORDER BY avgRating DESC
LIMIT 15
```
The `UNWIND p.tags AS tagName` step is the key move: it turns a **list property into rows**, so
each element can become its own node. This is the standard recipe for de-listing any array
property.

</details>

**Exercise 10.2** — Add a `:Dormant` label to every customer whose most recent order is more than
400 days older than the newest order in the database. Then count them by segment.

<details>
<summary><b>&#128161; Show solution &mdash; 10.2</b></summary>

```cypher
// Find the dataset's "today"
MATCH (o:Order)
WITH max(o.orderDate) AS latest

MATCH (c:Customer)-[:PLACED]->(o2:Order)
WITH c, latest, max(o2.orderDate) AS lastOrder
WHERE duration.inDays(lastOrder, latest).days > 400
SET c:Dormant, c.daysSinceLastOrder = duration.inDays(lastOrder, latest).days
RETURN count(c) AS dormantCustomers;

// Then:
MATCH (c:Dormant)
RETURN c.segment AS segment, count(*) AS customers,
       round(avg(c.daysSinceLastOrder)) AS avgDaysDormant
ORDER BY customers DESC
```
`duration.inDays(a, b)` returns a duration; `.days` extracts the whole-day component.
Anchoring on `max(o.orderDate)` rather than `datetime()` matters here — this is historical data,
so "now" should be the dataset's own latest timestamp, not the real clock.

</details>

**Exercise 10.3** *(harder)* — Materialise a `(:Product)-[:FREQUENTLY_BOUGHT_WITH {support}]->(:Product)`
relationship for product pairs bought by 3 or more of the same customers. Then use it to build a
one-hop "complete the basket" recommendation.

<details>
<summary><b>&#128161; Show solution &mdash; 10.3</b></summary>

```cypher
// Materialise the co-purchase edges
MATCH (c:Customer)-[:PLACED]->(:Order)-[:CONTAINS]->(p1:Product)
MATCH (c)-[:PLACED]->(:Order)-[:CONTAINS]->(p2:Product)
WHERE p1.productId < p2.productId
WITH p1, p2, count(DISTINCT c) AS support
WHERE support >= 3
MERGE (p1)-[f:FREQUENTLY_BOUGHT_WITH]->(p2)
SET f.support = support, f.computedAt = datetime()
RETURN count(f) AS edgesCreated;

// One-hop basket completion
MATCH (p:Product {productId: $pid})-[f:FREQUENTLY_BOUGHT_WITH]-(other:Product)
RETURN other.name AS alsoBuy, other.price AS price, f.support AS support
ORDER BY support DESC
LIMIT 5
```
Note the **undirected match** `-[f:FREQUENTLY_BOUGHT_WITH]-` in the read query. We stored each
pair once, in one direction (because of the `<` trick), so reading it undirected means we find
the edge whichever product the user is looking at. **Store directed, query undirected** — a very
common and very useful idiom.

</details>

## 10.7 Checkpoint 10

In [ ]:
br = gdb.run("MATCH (b:Brand) RETURN count(b) AS n")
sim = gdb.run("MATCH ()-[s:SIMILAR_TO]->() RETURN count(s) AS n")
onehop = gdb.run("""
    MATCH (me:Customer {customerId:$cid})-[s:SIMILAR_TO]-(peer:Customer)
    RETURN count(peer) AS n
""", {"cid": CID})

checkpoint("Part 10 - Writing and refactoring", [
    ("Brand nodes created",             int(br.iloc[0]["n"]) > 100),
    ("SIMILAR_TO edges materialised",   int(sim.iloc[0]["n"]) > 0),
    ("One-hop recommendation possible", int(onehop.iloc[0]["n"]) >= 0),
    ("I understand MERGE vs CREATE",    True),
])

---
# Part 11 — Modelling Deep Dive

Syntax you can look up. **Modelling is the skill that takes years.** This part is the one to
revisit after you've built something real.

## 11.1 Critique your own model

Let's audit the schema we built and ask what we'd do differently.

In [ ]:
# What does our schema actually look like now?
gdb.run("CALL db.schema.visualization()")

In [ ]:
# A more readable schema summary: which labels connect to which, and how densely.
gdb.run("""
    MATCH (a)-[r]->(b)
    RETURN labels(a)[0] AS fromLabel,
           type(r)      AS relationship,
           labels(b)[0] AS toLabel,
           count(*)     AS count
    ORDER BY count DESC
""")

### Audit findings

| Decision we made | Alternative | When the alternative wins |
|---|---|---|
| `Review` as a **node** | Relationship `(:Customer)-[:REVIEWED {rating}]->(:Product)` | If reviews never need their own connections — the relationship form is one hop shorter and much smaller |
| `SHIPPED_FROM` as a **relationship** | `(:Shipment)` node | The moment one order ships in several parcels, or you track scan events |
| `brand` as property **and** node | One or the other | Property only: simple filters. Node only: no sync burden |
| `Category` hierarchy via `PARENT_OF` | Materialised path property | If you *only* ever need the breadcrumb string, a property is faster to read |
| Order **status** as a property | A `:Cancelled` label | If you filter on one status constantly |
| No `Address` node | `(:Address)` node | Fraud, delivery-route optimisation, household resolution |

> **The uncomfortable truth: none of these is universally right.** A model that's excellent for a
> recommendation engine may be poor for a fraud platform on the same data. This is why graph
> modelling starts with **writing down the questions**, not with drawing boxes.

## 11.2 The modelling process that actually works

1. **Write the questions first.** Literally list them in English:
   *"Which suppliers would a recall affect?" "Which customers share a device?"*
2. **Underline the nouns.** Those are your candidate nodes.
3. **Underline the verbs.** Those are your candidate relationships.
4. **Walk each question through the model.** Trace the path with your finger. If a question needs
   more than 4–5 hops, or an aggregation to answer something structural, the model is wrong.
5. **Check the hubs.** Any node that will end up with millions of relationships (a
   `:Country`, a `:Status`) is a **supernode** and will wreck traversal performance.
6. **Iterate.** Graph schemas are cheap to change — that's a genuine advantage. Use it.

## 11.3 The supernode problem

A **supernode** is a node with a huge number of relationships. Traversing *through* one is
expensive because the engine must scan its relationship chain.

In [ ]:
# Find our hubs. Warehouses are our supernodes - only 5 of them, connected to everything.
gdb.run("""
    MATCH (n)
    WITH n, count { (n)--() } AS degree
    WHERE degree > 500
    RETURN labels(n)[0]                      AS label,
           coalesce(n.name, n.orderId, '?')  AS node,
           degree
    ORDER BY degree DESC
    LIMIT 15
""", show_time=True)

Our `Warehouse` nodes have thousands of relationships each. At our scale that's harmless. At
100 million orders it would be crippling — every query that traverses through a warehouse would
scan a colossal relationship chain.

### The three standard fixes

**1. Don't traverse through it.** Anchor elsewhere and use the supernode only as a filter:

```cypher
// Bad:  start at the warehouse and expand to millions of orders
MATCH (w:Warehouse {city:'Newark'})<-[:SHIPPED_FROM]-(o:Order)
WHERE o.orderDate > datetime('2024-01-01')

// Good: start at the selective end
MATCH (o:Order) WHERE o.orderDate > datetime('2024-01-01')
MATCH (o)-[:SHIPPED_FROM]->(w:Warehouse {city:'Newark'})
```

**2. Split the relationship type.** Instead of one `SHIPPED_FROM`, use
`SHIPPED_FROM_2024`, `SHIPPED_FROM_2025`. Neo4j stores relationship chains **per type and
direction**, so typing them narrows the scan dramatically.

**3. Introduce intermediate nodes.** Insert a fan-out layer:
`(:Warehouse)-[:HANDLED]->(:ShippingBatch)-[:INCLUDES]->(:Order)`.

## 11.4 Modelling time

Time is the most commonly mis-modelled dimension in graphs. Three approaches:

| Approach | Model | Best for |
|---|---|---|
| **Property** (what we used) | `o.orderDate = datetime(...)` | Filtering, ranges, aggregation |
| **Time tree** | `(:Year)-[:HAS_MONTH]->(:Month)-[:HAS_DAY]->(:Day)<-[:ON]-(:Order)` | Navigating by calendar, sparse periods |
| **Linked list** | `(:Order)-[:NEXT_ORDER]->(:Order)` | Sequences: "what did they buy *next*?" |

**Use a property by default.** Time trees were common advice in the Neo4j 3.x era, before range
indexes on temporal types were good. Today they're usually unnecessary overhead.

The **linked list** is different — it genuinely enables queries that are painful otherwise.

In [ ]:
# Build a NEXT_ORDER chain per customer - this enables sequence analysis.
gdb.run("MATCH ()-[r:NEXT_ORDER]->() DELETE r")

gdb.run("""
    MATCH (c:Customer)-[:PLACED]->(o:Order)
    WITH c, o ORDER BY o.orderDate
    WITH c, collect(o) AS orders
    WHERE size(orders) > 1
    UNWIND range(0, size(orders) - 2) AS i
    WITH orders[i] AS current, orders[i + 1] AS next
    MERGE (current)-[r:NEXT_ORDER]->(next)
    SET r.daysBetween = duration.inDays(current.orderDate, next.orderDate).days
    RETURN count(r) AS chainLinks
""", show_time=True)

In [ ]:
# Now sequence questions are trivial: what do people buy AFTER a given product?
gdb.run("""
    MATCH (first:Order)-[:CONTAINS]->(trigger:Product)
    MATCH (first)-[n:NEXT_ORDER]->(second:Order)-[:CONTAINS]->(followUp:Product)
    WHERE trigger <> followUp
    WITH trigger, followUp, count(*) AS times, avg(n.daysBetween) AS avgDays
    WHERE times >= 2
    RETURN trigger.name  AS boughtThis,
           followUp.name AS thenBoughtThis,
           times,
           round(avgDays) AS avgDaysBetween
    ORDER BY times DESC
    LIMIT 15
""", show_time=True)

> **That query is genuinely hard in SQL.** You'd need a window function
> (`LEAD(order_id) OVER (PARTITION BY customer_id ORDER BY order_date)`) joined back to line
> items twice. With the linked list, "the next order" is a single relationship — and the
> `daysBetween` property gives you the timing for free.
>
> Note the cost: **the chain must be maintained on every write.** Insert a back-dated order and
> you must re-link its neighbours. That maintenance burden is the price of the read performance.

## 11.5 Exercises — Part 11

**Exercise 11.1** — Design a graph model for a **hospital**: patients, doctors, appointments,
diagnoses, prescriptions, medications, wards. Write the questions first, then the model. Identify
at least one place where you need an intermediate node, and one likely supernode.

<details>
<summary><b>&#128161; Show solution &mdash; 11.1</b></summary>

**Questions first:**
- Which patients did Dr X see in the last month?
- Which medications interact dangerously with what this patient already takes?
- Trace everyone exposed to an infected patient through shared wards.
- Which doctors over-prescribe a given drug relative to peers?

**Model:**
```
(:Patient)-[:HAD_APPOINTMENT]->(:Appointment)-[:WITH_DOCTOR]->(:Doctor)
(:Appointment)-[:RESULTED_IN]->(:Diagnosis)-[:IS_CONDITION]->(:Condition)
(:Appointment)-[:PRODUCED]->(:Prescription)-[:FOR_MEDICATION]->(:Medication)
(:Medication)-[:INTERACTS_WITH {severity}]->(:Medication)
(:Patient)-[:ADMITTED_TO {from, to}]->(:Ward)
(:Doctor)-[:WORKS_IN]->(:Department)
```

**Intermediate node:** `Appointment` and `Prescription` are both reified events. A prescription
inherently involves doctor + patient + medication + date — four entities, so it cannot be one
relationship.

**Supernodes:** `(:Condition {name:'Hypertension'})` and common `(:Medication)` nodes will each
attract millions of edges. Mitigation: never traverse *from* the condition; anchor on the patient
or a date-bounded appointment and use the condition as a filter.

**Note the exposure query** — `(:Patient)-[:ADMITTED_TO]->(:Ward)<-[:ADMITTED_TO]-(:Patient)`
with overlapping date ranges. That's contact tracing, and it is exactly the shape that made
graph databases prominent during COVID.

</details>

**Exercise 11.2** — Our `Review` node currently has no connection to the `Order` that produced
it, so we can't verify a review corresponds to a real purchase *of that order*. Add a
`(:Review)-[:FOR_ORDER]->(:Order)` relationship where one can be inferred, and report how many
reviews you could and could not link.

<details>
<summary><b>&#128161; Show solution &mdash; 11.2</b></summary>

```cypher
// Link a review to the order in which that customer bought that product.
// If several orders qualify, take the most recent one before the review date.
MATCH (c:Customer)-[:WROTE]->(r:Review)-[:REVIEWS]->(p:Product)
OPTIONAL MATCH (c)-[:PLACED]->(o:Order)-[:CONTAINS]->(p)
WHERE o.orderDate <= r.reviewDate
WITH r, o ORDER BY o.orderDate DESC
WITH r, head(collect(o)) AS bestOrder
FOREACH (_ IN CASE WHEN bestOrder IS NULL THEN [] ELSE [1] END |
    MERGE (r)-[:FOR_ORDER]->(bestOrder))
RETURN count(*) AS reviewsProcessed;

// Report coverage
MATCH (r:Review)
RETURN count(*) AS totalReviews,
       count { (r)-[:FOR_ORDER]->(:Order) } AS linked
```
The `FOREACH (_ IN CASE WHEN ... THEN [] ELSE [1] END | ...)` idiom is Cypher's **conditional
write**. There is no `IF` statement, so you iterate over a list that is either empty (do nothing)
or single-element (do it once). Ugly, but it's the standard trick — and worth knowing, because
you'll hit it the first time you need a conditional `MERGE`.

</details>

## 11.6 Checkpoint 11

In [ ]:
chain = gdb.run("MATCH ()-[r:NEXT_ORDER]->() RETURN count(r) AS n")
hubs = gdb.run("""
    MATCH (n) WITH n, count { (n)--() } AS d
    WHERE d > 500 RETURN count(n) AS supernodes
""")

checkpoint("Part 11 - Modelling", [
    ("NEXT_ORDER chain built",        int(chain.iloc[0]["n"]) > 0),
    ("Supernodes identified",         int(hubs.iloc[0]["supernodes"]) > 0),
    ("I can critique a graph model",  True),
    ("I write questions before boxes", True),
])

---
# Part 12 — Capstone Project

## The brief

You are the data engineer for this e-commerce company. The **Head of Retention** comes to you:

> *"We're losing good customers and we don't know why. Build me something that finds customers
> who are about to churn, tells me why, and suggests what to offer them."*

## Requirements

Produce **one notebook section** containing:

### 1. A churn-risk model (graph features only)
For each customer, compute a churn-risk score from **at least five** signals you derive from the
graph. Suggested signals:
- Days since their last order, relative to their own historical purchase cadence
- Trend in order value (are recent orders smaller?)
- Ratio of returns or cancellations
- Whether they wrote a low-rated review recently
- Whether their similar customers (`SIMILAR_TO`) have also gone quiet
- Delivery experience: were their shipments late?

### 2. An explanation for each at-risk customer
Not just a score. A short list of the specific reasons, e.g.
`['Last order 380 days ago', '2 late deliveries', 'Rated a purchase 1 star']`.
Use **pattern comprehensions** to build this.

### 3. A personalised offer
For each at-risk customer, recommend 3 products using the `SIMILAR_TO` network built in Part 10,
excluding anything they already own, and weighted toward their preferred brands.

### 4. A visualisation
At least one chart or graph drawing that a non-technical stakeholder could read.

### 5. A written recommendation
150–300 words: what should the business actually do? Be specific about which customer group to
target first and why.

## Constraints

- **Everything in Cypher** where possible. Use pandas only for charting.
- **Parameterise** every query — no hard-coded ids.
- **`PROFILE` your heaviest query** and show that it uses an index.
- Handle nulls explicitly — a customer with no reviews must not vanish from your results.

## Marking guide

| Criterion | Weight |
|---|---|
| Correct, working Cypher | 30% |
| Uses graph structure (traversal, not just property filters) | 25% |
| Explanations are specific and derived from the data | 20% |
| Query performance considered and demonstrated | 15% |
| Clarity of the written recommendation | 10% |

In [ ]:
# ---------------------------------------------------------------------
# CAPSTONE STARTER - a scaffold to build on, not a solution.
# Extend the feature list, then write the explanation and offer sections.
# ---------------------------------------------------------------------

churn_features = gdb.run("""
    // Establish "today" as the dataset's most recent order
    MATCH (o:Order) WITH max(o.orderDate) AS today

    MATCH (c:Customer)-[:PLACED]->(o:Order)
    WITH today, c,
         count(o)                                                  AS orders,
         max(o.orderDate)                                          AS lastOrder,
         min(o.orderDate)                                          AS firstOrder,
         sum(o.totalAmount)                                        AS lifetimeValue,
         sum(CASE WHEN o.status = 'Returned'  THEN 1 ELSE 0 END)   AS returns,
         sum(CASE WHEN o.status = 'Cancelled' THEN 1 ELSE 0 END)   AS cancellations
    WHERE orders >= 2

    WITH c, orders, lifetimeValue, returns, cancellations, lastOrder,
         duration.inDays(lastOrder, today).days                    AS daysSinceLastOrder,
         duration.inDays(firstOrder, lastOrder).days / (orders - 1) AS avgDaysBetweenOrders

    // TODO (you): add late-delivery and negative-review signals here.
    //   Hint - pattern comprehensions keep this on one row per customer:
    //     size([(c)-[:PLACED]->(o:Order)-[s:SHIPPED_FROM]->() WHERE s.onTime = false | o]) AS lateDeliveries
    //     size([(c)-[:WROTE]->(r:Review) WHERE r.rating <= 2 | r])                          AS badReviews

    RETURN c.customerId       AS customerId,
           c.name             AS customer,
           c.segment          AS segment,
           orders,
           round(lifetimeValue, 2) AS lifetimeValue,
           daysSinceLastOrder,
           round(avgDaysBetweenOrders) AS avgDaysBetweenOrders,
           returns, cancellations,
           // a naive starting score - replace with your own
           round(
             (toFloat(daysSinceLastOrder) /
              CASE WHEN avgDaysBetweenOrders > 0
                   THEN avgDaysBetweenOrders ELSE 1 END) * 10
             + returns * 5 + cancellations * 3
           , 1) AS churnRisk
    ORDER BY churnRisk DESC
    LIMIT 25
""", show_time=True)

churn_features

---
# Appendix A — Cypher Cheat Sheet

### Reading

```cypher
MATCH (n:Label)                          // find nodes by label
MATCH (n:Label {prop: $value})           // inline equality filter
MATCH (a)-[:REL]->(b)                    // directed relationship
MATCH (a)-[:REL]-(b)                     // either direction
MATCH (a)-[r:REL]->(b) RETURN r.prop     // name it to read its properties
MATCH (a)-[:REL*1..4]->(b)               // variable length - ALWAYS bound it
MATCH p = shortestPath((a)-[*..8]-(b))   // shortest path (bind a and b first)
OPTIONAL MATCH (a)-[:REL]->(b)           // LEFT JOIN - fills nulls
```

### Filtering

```cypher
WHERE n.prop > 10 AND n.other IN ['a','b']
WHERE n.name STARTS WITH 'A'             // also ENDS WITH, CONTAINS
WHERE n.name =~ '(?i).*smith.*'          // regex, (?i) = case-insensitive
WHERE n.prop IS NULL / IS NOT NULL
WHERE (a)-[:REL]->(b)                    // pattern as a boolean
WHERE NOT (a)-[:REL]->(b)                // negated pattern (anti-join)
WHERE EXISTS { MATCH (a)-[:R]->(b) WHERE b.x > 1 }
WHERE size(n.list) > 2
WHERE any(x IN n.list WHERE x = 'y')     // also all / none / single
```

### Aggregating

```cypher
RETURN key, count(*), count(DISTINCT x)  // grouping is IMPLICIT
RETURN sum(x), avg(x), min(x), max(x), stDev(x)
RETURN percentileCont(x, 0.5)            // median
RETURN collect(x)                        // gather into a list
RETURN count { (n)-[:REL]->() }          // count subquery - no row fan-out
WITH key, count(*) AS c WHERE c > 5      // this is HAVING
```

### Pipelines and subqueries

```cypher
WITH a, b WHERE ... ORDER BY ... LIMIT 10   // pipe to the next stage
CALL (x) { ... RETURN y }                   // subquery per row (5.23+)
CALL { WITH x ... RETURN y }                // pre-5.23 form
UNWIND $list AS row                         // list -> rows
UNION / UNION ALL                           // combine result sets
```

### Lists and expressions

```cypher
[x IN list WHERE cond | expr]            // list comprehension
[(a)-[:R]->(b) WHERE cond | b.name]      // PATTERN comprehension
reduce(t = 0, x IN list | t + x)         // fold
list[0..3]  head(list)  last(list)  size(list)  range(1,10)  reverse(list)
CASE WHEN cond THEN a ELSE b END
coalesce(a, b, 0)                        // first non-null
```

### Writing

```cypher
CREATE (n:Label {prop: 1})               // always creates
MERGE  (n:Label {key: 1})                // upsert on the WHOLE pattern
  ON CREATE SET n.created = datetime()
  ON MATCH  SET n.seen = n.seen + 1
SET n.prop = 1,  n += $map,  n:NewLabel
REMOVE n.prop,  n:Label
DELETE r                                 // relationship
DETACH DELETE n                          // node + all its relationships
FOREACH (_ IN CASE WHEN cond THEN [1] ELSE [] END | ...)   // conditional write
```

### Schema

```cypher
CREATE CONSTRAINT c IF NOT EXISTS FOR (n:L) REQUIRE n.p IS UNIQUE
CREATE CONSTRAINT c IF NOT EXISTS FOR (n:L) REQUIRE n.p IS NOT NULL
CREATE INDEX i IF NOT EXISTS FOR (n:L) ON (n.p)
CREATE INDEX i IF NOT EXISTS FOR (n:L) ON (n.a, n.b)       // composite
CREATE TEXT INDEX / POINT INDEX / FULLTEXT INDEX ...
CREATE INDEX i IF NOT EXISTS FOR ()-[r:TYPE]-() ON (r.p)   // relationship
SHOW INDEXES  /  SHOW CONSTRAINTS
```

### Temporal and spatial

```cypher
datetime()  date()  localdatetime()  time()  duration({days: 7})
n.date.year / .month / .day / .quarter / .weekday / .hour
duration.inDays(a, b).days
point({latitude: 1.0, longitude: 2.0})
point.distance(p1, p2)                   // metres
```

### Introspection

```cypher
CALL db.schema.visualization()
CALL db.labels()  /  db.relationshipTypes()  /  db.propertyKeys()
EXPLAIN <query>                          // plan only, does not run
PROFILE <query>                          // runs, gives real db hits
```

---
# Appendix B — Aura Free limits and what's missing

| Limit | Value |
|---|---|
| Nodes | 200,000 |
| Relationships | 400,000 |
| Instances | 1 |
| Auto-pause | after 3 days idle (resumes on connect) |
| APOC | **Core only** (`apoc.coll.*`, `apoc.text.*`, `apoc.map.*` — no `apoc.load.*`, no `apoc.periodic.*`) |
| Graph Data Science (GDS) | **Not available** |
| `LOAD CSV` from `file:///` | Not available — public HTTPS URLs only |
| Multiple databases | Not available — single `neo4j` database |

**Our graph after Part 10:** roughly 18,000 nodes and 45,000 relationships — about 9% of the node
budget and 11% of the relationship budget. Plenty of room to keep experimenting.

### Where to go next

| Topic | Why | Where |
|---|---|---|
| **Graph Data Science** | PageRank, Louvain, node embeddings, link prediction | Neo4j GDS library — needs AuraDS or self-hosted |
| **Neo4j Browser** | Interactive visual exploration | Already in your Aura console |
| **Bloom** | Visual exploration for non-technical users | Aura console |
| **APOC** | Utility procedures for data manipulation | `CALL apoc.help('coll')` on your instance |
| **GraphQL** | Auto-generate an API from your graph | Neo4j GraphQL Library |
| **LangChain / GraphRAG** | Knowledge graphs for LLM retrieval | `Neo4jGraph` + `GraphCypherQAChain` |
| **Cypher certification** | Free, and a genuinely good revision exercise | GraphAcademy |

---
# Appendix C — Course wrap-up

In [ ]:
# Final state of your graph
summary = gdb.run("""
    MATCH (n)
    UNWIND labels(n) AS label
    RETURN label, count(*) AS nodes
    ORDER BY nodes DESC
""")
rels = gdb.run("""
    MATCH ()-[r]->()
    RETURN type(r) AS relationship, count(*) AS count
    ORDER BY count DESC
""")

total_nodes = int(gdb.run("MATCH (n) RETURN count(n) AS n").iloc[0]["n"])
total_rels = int(gdb.run("MATCH ()-[r]->() RETURN count(r) AS n").iloc[0]["n"])

print("=" * 62)
print("YOUR GRAPH")
print("=" * 62)
print("Nodes:         {:>8,}   ({:.1%} of the Aura Free limit)".format(
    total_nodes, total_nodes / 200000))
print("Relationships: {:>8,}   ({:.1%} of the Aura Free limit)".format(
    total_rels, total_rels / 400000))
print()
display(summary)
display(rels)

In [ ]:
# Final checkpoint - the whole course
final = [
    ("Connected to Aura",              total_nodes > 0),
    ("All 7 core labels loaded",       len(summary) >= 7),
    ("Payment nodes added (Part 8)",
     "Payment" in summary["label"].values),
    ("Brand nodes created (Part 10)",
     "Brand" in summary["label"].values),
    ("Device fraud ring seeded",
     "Device" in summary["label"].values),
    ("SIMILAR_TO materialised",
     "SIMILAR_TO" in rels["relationship"].values),
    ("NEXT_ORDER chain built",
     "NEXT_ORDER" in rels["relationship"].values),
]
checkpoint("FINAL - Complete course", final)

print("""
What you can now do
-------------------
  * Explain when a graph database beats a relational one, with evidence
  * Model a domain as nodes, relationships, labels and properties
  * Load data into Aura safely and quickly with batched UNWIND
  * Write Cypher from simple filters to multi-stage analytical pipelines
  * Traverse variable-depth hierarchies and find shortest paths
  * Build recommendation engines and fraud detection in pure Cypher
  * Read a query plan and fix a slow query
  * Refactor a live graph without downtime

Next: pick a dataset you actually care about and model it yourself.
That is the only way the modelling instinct develops.
""")

In [ ]:
# Close the driver when you are finished.
# (Re-run the connection cell in Part 0 if you want to continue afterwards.)
gdb.close()
print("Connection closed. Well done.")